# Previsão de evasão na Rede Federal do Nordeste, com análise de viés

**Maria Ingrid Xavier Sousa** · Projeto final de Inteligência Artificial · Ciência da Computação, UFPI, 2026.2 ·
prof. Pedro Almir

**Pergunta de pesquisa.** Com informações disponíveis na matrícula, é possível estimar o risco de evasão de
alunos de cursos técnicos e de graduação da Rede Federal do Nordeste melhor do que a taxa histórica do curso, e o
modelo erra de forma igual entre grupos de estudantes (sexo, cor/raça, renda declarada, idade)?

**Tipo de aprendizado.** Supervisionado: classificação binária (evadiu ou concluiu), com probabilidades calibradas.

**Problema e relevância.** Mais da metade das matrículas encerradas nos Institutos Federais do Nordeste termina em
evasão (58,4% nesta base). Identificar cedo os perfis com maior risco ajuda equipes pedagógicas e de assistência
estudantil a priorizar acompanhamento desde o ingresso. **Público beneficiado:** gestores, equipes pedagógicas e
de assistência estudantil e os próprios estudantes. **Limites do escopo:** o modelo descreve o histórico de alunos
com perfil parecido, não prevê o destino de uma pessoa; usa só dados do ingresso; cobre o Nordeste, edições 2021 a
2023, cursos técnicos e de graduação; e não deve ser usado para negar vagas, bolsas ou direitos.

Este notebook reproduz o trabalho inteiro, dos microdados brutos aos resultados do artigo:

1. Dados e histórico do pré-processamento (v1 com Matrículas, v2 com Eficiência Acadêmica)
2. Junção das edições 2021–2023, padronização dos nomes e divisão 70/20/10
3. Análise exploratória (só treino)
4. Modelagem: baselines, gradient boosting, calibração e teste
5. Análise de viés por grupo
6. Experimentos de melhoria e nível de confiança
7. Renda e cor/raça: registro no ingresso, indicadores e cor/raça como entrada
8. Modelo do Nordeste x modelo só do IFPI
9. Retidos: análise de sensibilidade

**Links:** [site](https://mingridxs.github.io/evasao-rede-federal-ne/) ·
[notebook no Kaggle](https://www.kaggle.com/code/mariaingridxavier/evasao-na-rede-federal-do-ne-previsao-e-vies) ·
[repositório](https://github.com/mingridxs/evasao-rede-federal-ne) ·
[dataset no Kaggle](https://www.kaggle.com/datasets/mariaingridxavier/evasao-na-rede-federal-do-nordeste-pnp-2021-2023) ·
dados: Plataforma Nilo Peçanha (MEC), licença ODbL.

**Como rodar no Kaggle:** anexe o dataset com os microdados (*Add Input*) e use *Run All*.
Leva cerca de 20 a 30 minutos na CPU padrão. **Localmente:** coloque os brutos em `dados/brutos/`
com os nomes `microdados_eficiencia_academica_ANO.csv` e `microdados_matriculas_ANO.csv`.

In [ ]:
import glob
import os
import shutil
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

NO_KAGGLE = Path("/kaggle/input").exists()
if NO_KAGGLE:
    os.chdir("/kaggle/working")
for pasta in ["dados/brutos", "dados/processados", "resultados", "figuras"]:
    Path(pasta).mkdir(parents=True, exist_ok=True)

# No Kaggle, os arquivos do dataset ficam em /kaggle/input/...; aqui eles ganham
# atalhos com os nomes que o código usa.
if NO_KAGGLE:
    for arq in glob.glob("/kaggle/input/**/*.csv", recursive=True):
        nome = Path(arq).name
        if nome.startswith("microdados_") and not Path("dados/brutos", nome).exists():
            os.symlink(arq, Path("dados/brutos", nome))
        elif nome == "ef_2021_2023.csv" and not Path("dados/processados", nome).exists():
            shutil.copy(arq, Path("dados/processados", nome))

EDICOES = [2021, 2022, 2023]
BRUTOS_EF = {ano: Path(f"dados/brutos/microdados_eficiencia_academica_{ano}.csv") for ano in EDICOES}
BRUTO_MAT = Path("dados/brutos/microdados_matriculas_2023.csv")
TEM_BRUTOS = all(p.exists() for p in BRUTOS_EF.values())
print("Python", sys.version.split()[0], "· pandas", pd.__version__, "· scikit-learn", sklearn.__version__,
      "· matplotlib", matplotlib.__version__)
print("Brutos de Eficiência Acadêmica:", {a: p.exists() for a, p in BRUTOS_EF.items()})
print("Bruto de Matrículas 2023:", BRUTO_MAT.exists())
if not TEM_BRUTOS:
    assert Path("dados/processados/ef_2021_2023.csv").exists(), \
        "Sem os brutos e sem a base limpa: anexe o dataset do Kaggle ou coloque os arquivos em dados/brutos/."
    print("Sem os brutos: o pré-processamento é pulado e o notebook usa a base limpa do dataset.")

## 1. Dados e pré-processamento

Microdados da [Plataforma Nilo Peçanha](https://dadosabertos.mec.gov.br/pnp) (PNP), do MEC, sob a
Open Data Commons Open Database License (ODbL). O recorte é o Nordeste, em cursos técnicos e de graduação
(tecnologia, licenciatura e bacharelado).

### 1.1 Histórico: a versão 1 do pré-processamento

A versão 1 foi testada primeiro com os microdados de **Matrículas 2023**, que trazem a situação de cada aluno
**no ano-base**. O problema apareceu nesse teste: quem evade cedo já saiu da base, então o arquivo mostra
sobretudo quem ainda está estudando, e ele não diz o desfecho final de quem saiu antes. Por isso a fonte passou a
ser a **Eficiência Acadêmica**, e a mesma versão 1 foi aplicada às edições 2021, 2022 e 2023.

A versão 2 (seção 1.2) acrescentou as colunas de demanda (inscritos por vaga), que só existem no arquivo de
Matrículas, e a opção de guardar os retidos. Nas edições de Eficiência Acadêmica, ela gera as mesmas bases que a
versão 1, o que é conferido no fim da seção 1.2.

In [ ]:
import re
import unicodedata

import pandas as pd

# Atributos conhecidos no ingresso (nomes já normalizados).
ATRIBUTOS = [
    "instituicao", "unidade_de_ensino", "uf", "regiao",
    "nome_de_curso", "tipo_de_curso", "tipo_de_oferta", "modalidade_de_ensino",
    "eixo_tecnologico", "subeixo_tecnologico", "turno", "fonte_de_financiamento",
    "carga_horaria",
    "sexo", "cor_raca", "renda_familiar", "idade",
]
DATAS = ["data_de_inicio_do_ciclo", "data_de_fim_previsto_do_ciclo",
         "data_de_ocorrencia_da_matricula"]
# No arquivo de 2023 a coluna 'Região' vem preenchida com o nome da instituição,
# então a região é derivada da UF.
REGIAO_POR_UF = {
    **dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
    **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
    **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
    **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
    **dict.fromkeys(["PR", "RS", "SC"], "Sul"),
}
# Cursos regulares de nível técnico e de graduação. Ficam fora, por padrão,
# qualificação (FIC), pós-graduação e ensino médio, em que a evasão é outro fenômeno.
TIPOS_PADRAO = ["Técnico", "Tecnologia", "Licenciatura", "Bacharelado"]
ROTULO = "categoria_da_situacao"
ID = "codigo_da_matricula"

# Ficam de fora de propósito:
# - situacao_de_matricula e mes_de_ocorrencia_da_situacao: revelam o rótulo (vazamento);
# - matricula_atendida, numero_de_registros, fator_esforco_curso, vagas_*, total_de_inscritos:
#   indicadores de gestão, não características do aluno;
# - códigos (co_inst, cod_unidade, SISTEC, município): redundantes com os nomes;
# - faixa_etaria: redundante com idade.


def normaliza(nome: str) -> str:
    """'Cor / Raça' -> 'cor_raca'."""
    sem_acento = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sem_acento.lower()).strip("_")


def le_csv(caminho: str) -> pd.DataFrame:
    """Lê o CSV detectando codificação e separador."""
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(caminho, encoding=enc) as f:
                cabecalho = f.readline()
            sep = max([";", "\t", ","], key=cabecalho.count)
            return pd.read_csv(caminho, sep=sep, encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não consegui ler {caminho}")


def sem_acento(serie: pd.Series) -> pd.Series:
    """Minúsculas e sem acento, para comparar 'Piauí' com 'piaui'."""
    return (serie.fillna("").str.normalize("NFKD")
            .str.encode("ascii", "ignore").str.decode("ascii").str.lower())


def preprocessa_v1(arquivos: dict, regiao: str | None = None, uf: str | None = None,
                instituicao: str | None = None, tipos_de_curso: list | None = TIPOS_PADRAO,
                so_ciclos_encerrados: bool = True) -> pd.DataFrame:
    """
    arquivos: {ano_base: caminho_do_csv}
    regiao / uf / instituicao: recortes opcionais e combináveis
        (ex.: regiao="Nordeste", uf="PI", instituicao="Piauí")
    tipos_de_curso: lista de tipos mantidos; None mantém todos
    so_ciclos_encerrados: mantém só matrículas cujo fim previsto do ciclo já passou
        no último ano-base carregado, para não comparar quem evadiu cedo com quem
        ainda nem teria tido tempo de concluir.
    """
    partes = []
    for ano_base, caminho in arquivos.items():
        df = le_csv(caminho)
        df.columns = [normaliza(c) for c in df.columns]
        # Usa o ano-base gravado no próprio arquivo (coluna 'Ano'), se existir.
        ano_arquivo = pd.to_numeric(df.get("ano"), errors="coerce") if "ano" in df.columns else None
        if ano_arquivo is not None and ano_arquivo.notna().all():
            anos = sorted(ano_arquivo.astype(int).unique())
            if anos != [int(ano_base)]:
                print(f"AVISO - {caminho}: informado {ano_base}, coluna 'Ano' traz {[int(a) for a in anos]}; "
                      "usando o valor do arquivo")
            df["ano_base"] = ano_arquivo.astype(int)
        else:
            df["ano_base"] = int(ano_base)
        partes.append(df)
    df = pd.concat(partes, ignore_index=True)
    print(f"Linhas lidas: {len(df):,}")

    # Confere se as colunas esperadas existem.
    esperadas = ATRIBUTOS + DATAS + [ROTULO]
    faltando = [c for c in esperadas if c not in df.columns]
    if ROTULO in faltando:
        raise KeyError(f"Coluna de rótulo ausente. Colunas do arquivo: {list(df.columns)}")
    if faltando:
        print(f"AVISO - colunas não encontradas (ignoradas): {faltando}")
    atributos = [c for c in ATRIBUTOS if c in df.columns]
    datas = [c for c in DATAS if c in df.columns]

    # Limpa espaços e padroniza vazios.
    for c in atributos + [ROTULO]:
        df[c] = df[c].str.strip().replace({"": pd.NA, "-": pd.NA})

    # Recortes.
    if "uf" in df.columns:
        df["uf"] = df["uf"].str.upper()
        df["regiao"] = df["uf"].map(REGIAO_POR_UF)
    if regiao:
        df = df[sem_acento(df["regiao"]).map(normaliza) == normaliza(regiao)]
    if uf and "uf" in df.columns:
        df = df[df["uf"] == uf.upper()]
    if instituicao and "instituicao" in df.columns:
        alvo = normaliza(instituicao).replace("_", " ")
        df = df[sem_acento(df["instituicao"]).str.contains(alvo, regex=False)]
    if tipos_de_curso and "tipo_de_curso" in df.columns:
        df = df[df["tipo_de_curso"].isin(tipos_de_curso)]
    print(f"Após recorte: {len(df):,}")

    # Duplicatas: mesma matrícula no mesmo ano-base.
    if ID in df.columns:
        df = df.drop_duplicates(subset=[ID, "ano_base"])

    # Transferência interna (mudou de curso na mesma instituição) não é evasão.
    # (A PNP conta como evasão; aqui é uma decisão do trabalho, registrada na metodologia.)
    if "situacao_de_matricula" in df.columns:
        df = df[~sem_acento(df["situacao_de_matricula"]).str.contains("interna", regex=False)]

    # Rótulo: só matrículas com desfecho. 'Em Curso' sai (no arquivo de Eficiência
    # Acadêmica, são alunos ainda ativos depois do fim previsto do ciclo: retidos).
    # 'Concluídos' (Concluída) e 'Concluintes' (Integralizada) contam como não evasão.
    cat = df[ROTULO].str.lower().fillna("")
    df = df[cat.str.startswith(("evadid", "conclu"))].copy()
    df["evadiu"] = df[ROTULO].str.lower().str.startswith("evadid").astype(int)
    print(f"Com desfecho (evadido ou concluinte): {len(df):,}")

    # Com vários anos-base, a mesma matrícula pode aparecer mais de uma vez:
    # fica o registro mais recente.
    if ID in df.columns and df["ano_base"].nunique() > 1:
        repetidas = df[ID].duplicated().sum()
        df = df.sort_values("ano_base").drop_duplicates(subset=[ID], keep="last")
        print(f"Matrículas repetidas entre anos (removidas): {repetidas:,}")

    # Datas e atributos derivados.
    for c in datas:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors="coerce")
    if "data_de_inicio_do_ciclo" in df.columns:
        df["ano_ingresso"] = df["data_de_inicio_do_ciclo"].dt.year
        df["mes_ingresso"] = df["data_de_inicio_do_ciclo"].dt.month
    if {"data_de_inicio_do_ciclo", "data_de_fim_previsto_do_ciclo"} <= set(df.columns):
        dias = (df["data_de_fim_previsto_do_ciclo"] - df["data_de_inicio_do_ciclo"]).dt.days
        df["duracao_prevista_meses"] = (dias / 30.44).round()
        if so_ciclos_encerrados:
            fim_ano = pd.Timestamp(year=int(df["ano_base"].max()), month=12, day=31)
            df = df[df["data_de_fim_previsto_do_ciclo"] <= fim_ano]
            print(f"Com ciclo já encerrado: {len(df):,}")

    # Numéricos.
    for c in ("idade", "carga_horaria"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False), errors="coerce")
    # Segundo as instruções dos microdados, 'idade' é a idade em 31/12 do ano-base.
    # Para usar só o que se sabe no ingresso, recua-se até o ano de ingresso.
    if {"idade", "ano_ingresso"} <= set(df.columns):
        df["idade_ingresso"] = df["idade"] - (df["ano_base"] - df["ano_ingresso"])
        df.loc[~df["idade_ingresso"].between(10, 90), "idade_ingresso"] = pd.NA
        atributos = [c for c in atributos if c != "idade"]
        derivados_idade = ["idade_ingresso"]
    else:
        derivados_idade = []

    # Categóricas: 'Não declarada' fica como categoria própria (a ausência é informativa).
    categoricas = [c for c in atributos if c != "carga_horaria"]
    df[categoricas] = df[categoricas].fillna("Não informado")

    derivados = [c for c in ("ano_ingresso", "mes_ingresso", "duracao_prevista_meses")
                 if c in df.columns] + derivados_idade
    final = df[["ano_base"] + atributos + derivados + ["evadiu"]].reset_index(drop=True)

    print(f"\nBase final: {final.shape[0]:,} linhas x {final.shape[1]} colunas")
    print(f"Taxa de evasão: {final['evadiu'].mean():.1%}")
    if "tipo_de_curso" in final.columns:
        print("\nPor tipo de curso (matrículas e taxa de evasão):")
        print(final.groupby("tipo_de_curso")["evadiu"].agg(["size", "mean"])
              .sort_values("size", ascending=False).round(3).to_string())
    print("\nValores ausentes por coluna:")
    print(final.isna().sum()[lambda s: s > 0].to_string() or "  nenhum")
    return final

In [ ]:
if BRUTO_MAT.exists():
    print("========== v1 com Matrículas 2023 (teste que levou à troca de fonte) ==========")
    v1_mat = preprocessa_v1({2023: str(BRUTO_MAT)}, regiao="Nordeste")
    print(f"\nv1 (Matrículas 2023): {len(v1_mat):,} matrículas com desfecho, evasão de {v1_mat['evadiu'].mean():.1%}")
else:
    print("Arquivo de Matrículas 2023 não encontrado: o teste da v1 com Matrículas não roda aqui.")

v1_ef = {}
if TEM_BRUTOS:
    for ano, caminho in BRUTOS_EF.items():
        print(f"\n========== v1 com Eficiência Acadêmica {ano} ==========")
        v1_ef[ano] = preprocessa_v1({ano: str(caminho)}, regiao="Nordeste")

### 1.2 Versão 2: Eficiência Acadêmica

Os microdados de **Eficiência Acadêmica** trazem, para cada turma cujo prazo previsto terminou no ano anterior
ao da edição, a situação final de cada aluno: concluiu, evadiu ou continua em curso (retido). É exatamente o
desfecho que o modelo precisa prever. Por isso a v2 usa as edições 2021, 2022 e 2023 desse arquivo.

Decisões da v2, registradas na documentação do projeto:
- a região vem da UF (a coluna "Região" da PNP traz outra coisa em algumas edições);
- transferência interna (troca de curso na mesma instituição) não conta como evasão;
- a idade no ingresso é recalculada a partir da idade em 31/12 do ano-base;
- "não declarado" vira categoria própria;
- os **retidos** saem da base principal, porque o desfecho deles ainda não existe. Eles são guardados à parte
  para a análise de sensibilidade da seção 9.

In [ ]:
import re
import unicodedata

import pandas as pd

# Atributos conhecidos no ingresso (nomes já normalizados).
ATRIBUTOS = [
    "instituicao", "unidade_de_ensino", "uf", "regiao",
    "nome_de_curso", "tipo_de_curso", "tipo_de_oferta", "modalidade_de_ensino",
    "eixo_tecnologico", "subeixo_tecnologico", "turno", "fonte_de_financiamento",
    "carga_horaria",
    "sexo", "cor_raca", "renda_familiar", "idade",
]
DATAS = ["data_de_inicio_do_ciclo", "data_de_fim_previsto_do_ciclo",
         "data_de_ocorrencia_da_matricula"]
# No arquivo de 2023 a coluna 'Região' vem preenchida com o nome da instituição,
# então a região é derivada da UF.
REGIAO_POR_UF = {
    **dict.fromkeys(["AC", "AP", "AM", "PA", "RO", "RR", "TO"], "Norte"),
    **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
    **dict.fromkeys(["DF", "GO", "MT", "MS"], "Centro-Oeste"),
    **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
    **dict.fromkeys(["PR", "RS", "SC"], "Sul"),
}
# Cursos regulares de nível técnico e de graduação. Ficam fora, por padrão,
# qualificação (FIC), pós-graduação e ensino médio, em que a evasão é outro fenômeno.
TIPOS_PADRAO = ["Técnico", "Tecnologia", "Licenciatura", "Bacharelado"]
ROTULO = "categoria_da_situacao"
ID = "codigo_da_matricula"

# Ficam de fora de propósito:
# - situacao_de_matricula e mes_de_ocorrencia_da_situacao: revelam o rótulo (vazamento);
# - matricula_atendida, numero_de_registros, fator_esforco_curso, vagas_*, total_de_inscritos:
#   indicadores de gestão, não características do aluno;
# - códigos (co_inst, cod_unidade, SISTEC, município): redundantes com os nomes;
# - faixa_etaria: redundante com idade.


def normaliza(nome: str) -> str:
    """'Cor / Raça' -> 'cor_raca'."""
    sem_acento = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sem_acento.lower()).strip("_")


def le_csv(caminho: str) -> pd.DataFrame:
    """Lê o CSV detectando codificação e separador."""
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(caminho, encoding=enc) as f:
                cabecalho = f.readline()
            sep = max([";", "\t", ","], key=cabecalho.count)
            return pd.read_csv(caminho, sep=sep, encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não consegui ler {caminho}")


def sem_acento(serie: pd.Series) -> pd.Series:
    """Minúsculas e sem acento, para comparar 'Piauí' com 'piaui'."""
    return (serie.fillna("").str.normalize("NFKD")
            .str.encode("ascii", "ignore").str.decode("ascii").str.lower())


def preprocessa(arquivos: dict, regiao: str | None = None, uf: str | None = None,
                instituicao: str | None = None, tipos_de_curso: list | None = TIPOS_PADRAO,
                so_ciclos_encerrados: bool = True, manter_retidos: bool = False) -> pd.DataFrame:
    """
    arquivos: {ano_base: caminho_do_csv}
    regiao / uf / instituicao: recortes opcionais e combináveis
        (ex.: regiao="Nordeste", uf="PI", instituicao="Piauí")
    tipos_de_curso: lista de tipos mantidos; None mantém todos
    so_ciclos_encerrados: mantém só matrículas cujo fim previsto do ciclo já passou
        no último ano-base carregado, para não comparar quem evadiu cedo com quem
        ainda nem teria tido tempo de concluir.
    manter_retidos: mantém também os alunos 'Em curso' depois do fim previsto
        (retidos), com evadiu = 0 e a coluna 'retido' = 1. Usado só na análise
        de sensibilidade (scripts/15_retidos.py); a base principal não os inclui.
    """
    partes = []
    for ano_base, caminho in arquivos.items():
        df = le_csv(caminho)
        df.columns = [normaliza(c) for c in df.columns]
        # Usa o ano-base gravado no próprio arquivo (coluna 'Ano'), se existir.
        ano_arquivo = pd.to_numeric(df.get("ano"), errors="coerce") if "ano" in df.columns else None
        if ano_arquivo is not None and ano_arquivo.notna().all():
            anos = sorted(ano_arquivo.astype(int).unique())
            if anos != [int(ano_base)]:
                print(f"AVISO - {caminho}: informado {ano_base}, coluna 'Ano' traz {[int(a) for a in anos]}; "
                      "usando o valor do arquivo")
            df["ano_base"] = ano_arquivo.astype(int)
        else:
            df["ano_base"] = int(ano_base)
        partes.append(df)
    df = pd.concat(partes, ignore_index=True)
    print(f"Linhas lidas: {len(df):,}")

    # Confere se as colunas esperadas existem.
    esperadas = ATRIBUTOS + DATAS + [ROTULO]
    faltando = [c for c in esperadas if c not in df.columns]
    if ROTULO in faltando:
        raise KeyError(f"Coluna de rótulo ausente. Colunas do arquivo: {list(df.columns)}")
    if faltando:
        print(f"AVISO - colunas não encontradas (ignoradas): {faltando}")
    atributos = [c for c in ATRIBUTOS if c in df.columns]
    datas = [c for c in DATAS if c in df.columns]

    # Limpa espaços e padroniza vazios.
    for c in atributos + [ROTULO]:
        df[c] = df[c].str.strip().replace({"": pd.NA, "-": pd.NA})

    # Recortes.
    if "uf" in df.columns:
        df["uf"] = df["uf"].str.upper()
        df["regiao"] = df["uf"].map(REGIAO_POR_UF)
    if regiao:
        df = df[sem_acento(df["regiao"]).map(normaliza) == normaliza(regiao)]
    if uf and "uf" in df.columns:
        df = df[df["uf"] == uf.upper()]
    if instituicao and "instituicao" in df.columns:
        alvo = normaliza(instituicao).replace("_", " ")
        df = df[sem_acento(df["instituicao"]).str.contains(alvo, regex=False)]
    if tipos_de_curso and "tipo_de_curso" in df.columns:
        df = df[df["tipo_de_curso"].isin(tipos_de_curso)]
    print(f"Após recorte: {len(df):,}")

    # Duplicatas: mesma matrícula no mesmo ano-base.
    if ID in df.columns:
        df = df.drop_duplicates(subset=[ID, "ano_base"])

    # Transferência interna (mudou de curso na mesma instituição) não é evasão.
    # (A PNP conta como evasão; aqui é uma decisão do trabalho, registrada na metodologia.)
    if "situacao_de_matricula" in df.columns:
        df = df[~sem_acento(df["situacao_de_matricula"]).str.contains("interna", regex=False)]

    # Rótulo: só matrículas com desfecho. 'Em Curso' sai (no arquivo de Eficiência
    # Acadêmica, são alunos ainda ativos depois do fim previsto do ciclo: retidos).
    # 'Concluídos' (Concluída) e 'Concluintes' (Integralizada) contam como não evasão.
    cat = df[ROTULO].str.lower().fillna("")
    mantidos = ("evadid", "conclu", "em curso") if manter_retidos else ("evadid", "conclu")
    df = df[cat.str.startswith(mantidos)].copy()
    df["evadiu"] = df[ROTULO].str.lower().str.startswith("evadid").astype(int)
    if manter_retidos:
        df["retido"] = df[ROTULO].str.lower().str.startswith("em curso").astype(int)
        print(f"Com desfecho ou retidos: {len(df):,} (retidos: {df['retido'].sum():,})")
    else:
        print(f"Com desfecho (evadido ou concluinte): {len(df):,}")

    # Com vários anos-base, a mesma matrícula pode aparecer mais de uma vez:
    # fica o registro mais recente.
    if ID in df.columns and df["ano_base"].nunique() > 1:
        repetidas = df[ID].duplicated().sum()
        df = df.sort_values("ano_base").drop_duplicates(subset=[ID], keep="last")
        print(f"Matrículas repetidas entre anos (removidas): {repetidas:,}")

    # Datas e atributos derivados.
    for c in datas:
        df[c] = pd.to_datetime(df[c], dayfirst=True, errors="coerce")
    if "data_de_inicio_do_ciclo" in df.columns:
        df["ano_ingresso"] = df["data_de_inicio_do_ciclo"].dt.year
        df["mes_ingresso"] = df["data_de_inicio_do_ciclo"].dt.month
    if {"data_de_inicio_do_ciclo", "data_de_fim_previsto_do_ciclo"} <= set(df.columns):
        dias = (df["data_de_fim_previsto_do_ciclo"] - df["data_de_inicio_do_ciclo"]).dt.days
        df["duracao_prevista_meses"] = (dias / 30.44).round()
        if so_ciclos_encerrados:
            fim_ano = pd.Timestamp(year=int(df["ano_base"].max()), month=12, day=31)
            df = df[df["data_de_fim_previsto_do_ciclo"] <= fim_ano]
            print(f"Com ciclo já encerrado: {len(df):,}")

    # Demanda do curso no ano-base: inscritos e vagas da fase inicial.
    # (Segundo o Guia PNP, os dois se referem ao ano-base da edição, não ao
    # ano de ingresso do aluno.)
    derivados_demanda = []
    if "total_de_inscritos" in df.columns:
        cols_vagas = [c for c in df.columns if c.startswith("vagas_")]
        df["inscritos"] = pd.to_numeric(df["total_de_inscritos"], errors="coerce")
        df["vagas"] = df[cols_vagas].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)
        df["inscritos_por_vaga"] = df["inscritos"] / df["vagas"].where(df["vagas"] > 0)
        derivados_demanda = ["inscritos", "vagas", "inscritos_por_vaga"]

    # Numéricos.
    for c in ("idade", "carga_horaria"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c].str.replace(",", ".", regex=False), errors="coerce")
    # Segundo as instruções dos microdados, 'idade' é a idade em 31/12 do ano-base.
    # Para usar só o que se sabe no ingresso, recua-se até o ano de ingresso.
    if {"idade", "ano_ingresso"} <= set(df.columns):
        df["idade_ingresso"] = df["idade"] - (df["ano_base"] - df["ano_ingresso"])
        df.loc[~df["idade_ingresso"].between(10, 90), "idade_ingresso"] = pd.NA
        atributos = [c for c in atributos if c != "idade"]
        derivados_idade = ["idade_ingresso"]
    else:
        derivados_idade = []

    # Categóricas: 'Não declarada' fica como categoria própria (a ausência é informativa).
    categoricas = [c for c in atributos if c != "carga_horaria"]
    df[categoricas] = df[categoricas].fillna("Não informado")

    derivados = [c for c in ("ano_ingresso", "mes_ingresso", "duracao_prevista_meses")
                 if c in df.columns] + derivados_idade + derivados_demanda
    rotulos = ["evadiu"] + (["retido"] if manter_retidos else [])
    final = df[["ano_base"] + atributos + derivados + rotulos].reset_index(drop=True)

    print(f"\nBase final: {final.shape[0]:,} linhas x {final.shape[1]} colunas")
    print(f"Taxa de evasão: {final['evadiu'].mean():.1%}")
    if "tipo_de_curso" in final.columns:
        print("\nPor tipo de curso (matrículas e taxa de evasão):")
        print(final.groupby("tipo_de_curso")["evadiu"].agg(["size", "mean"])
              .sort_values("size", ascending=False).round(3).to_string())
    print("\nValores ausentes por coluna:")
    print(final.isna().sum()[lambda s: s > 0].to_string() or "  nenhum")
    return final

In [ ]:
if TEM_BRUTOS:
    for ano, caminho in BRUTOS_EF.items():
        print(f"\n========== Edição {ano} ==========")
        com_retidos = preprocessa({ano: str(caminho)}, regiao="Nordeste", manter_retidos=True)
        com_retidos.to_csv(f"dados/processados/ef_{ano}_com_retidos.csv", index=False)
        # A base principal é a mesma, sem os retidos e sem a coluna que os marca.
        principal = com_retidos[com_retidos["retido"] == 0].drop(columns="retido")
        principal.to_csv(f"dados/processados/ef_{ano}.csv", index=False)
        if ano in v1_ef:
            igual = v1_ef[ano].reset_index(drop=True).equals(principal.reset_index(drop=True))
            print(f"v1 e v2 geram a mesma base em {ano}: {'sim' if igual else 'NÃO, conferir'}")
        print(f"Base principal de {ano}: {len(principal):,} matrículas · retidos guardados à parte: "
              f"{int(com_retidos['retido'].sum()):,}")

## 2. Junção das edições, padronização dos nomes e divisão

A edição 2023 traz o nome completo das instituições ("Instituto Federal do Piauí"), e as de 2021 e 2022, a
sigla ("IFPI"). Sem padronizar, o modelo trataria a mesma instituição como duas, e o nome passaria a indicar a
edição. A função abaixo também é usada na seção 9.

In [ ]:
import pandas as pd

NOME_INSTITUICAO = {
    "IF BAIANO": "Instituto Federal Baiano",
    "IF SERTÃO-PE": "Instituto Federal do Sertão Pernambucano",
    "IFAL": "Instituto Federal de Alagoas",
    "IFBA": "Instituto Federal da Bahia",
    "IFCE": "Instituto Federal do Ceará",
    "IFMA": "Instituto Federal do Maranhão",
    "IFPB": "Instituto Federal da Paraíba",
    "IFPE": "Instituto Federal de Pernambuco",
    "IFPI": "Instituto Federal do Piauí",
    "IFRN": "Instituto Federal do Rio Grande do Norte",
    "IFS": "Instituto Federal de Sergipe",
    "ESUFRN": "Escola de Saúde da UFRN",
    "UFAL-ETA": "Escola Técnica de Artes da UFAL",
    "UFCG-ETSC": "Escola Técnica de Saúde de Cajazeiras da UFCG",
    "UFMA-COLUN": "Colégio Universitário da UFMA",
    "UFPB-CAVN": "Colégio Agrícola Vidal de Negreiros da UFPB",
    "UFPB-ETS": "Escola Técnica de Saúde da UFPB",
    "UFPB-ESTES": "Escola Técnica de Saúde da UFPB",
    "UFPI-CABJ": "Colégio Técnico de Bom Jesus da UFPI",   # antigo Colégio Agrícola de Bom Jesus
    "UFPI-CTBJ": "Colégio Técnico de Bom Jesus da UFPI",
    "UFPI-CAT": "Colégio Técnico de Teresina da UFPI",     # antigo Colégio Agrícola de Teresina
    "UFPI-CTF": "Colégio Técnico de Floriano da UFPI",
    "UFRN-EAJ": "Escola Agrícola de Jundiaí da UFRN",
    "UFRN-MÚSICA": "Escola de Música da UFRN",
    "UFRN-MUSICA": "Escola de Música da UFRN",
    "UFRPE-CODAI": "Colégio Agrícola Dom Agostinho Ikas da UFRPE",
}


def padroniza_nomes(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    df = df.copy()
    antes = df["instituicao"].nunique()
    sigla = df["instituicao"].str.strip().str.replace(r"\s*-\s*", "-", regex=True).str.upper()
    df["instituicao"] = sigla.map({k.upper(): v for k, v in NOME_INSTITUICAO.items()}).fillna(df["instituicao"].str.strip())
    if verbose:
        print(f"Instituições: {antes} nomes antes, {df['instituicao'].nunique()} depois de padronizar.")
        por_edicao = pd.crosstab(df["instituicao"], df["ano_base"])
        incompletas = por_edicao[(por_edicao == 0).any(axis=1)]
        if len(incompletas):
            print("AVISO - instituições que não aparecem em todas as edições (confira se é outro nome a padronizar):")
            print(incompletas.to_string())
    for coluna in ["nome_de_curso", "unidade_de_ensino"]:
        chave = df[coluna].str.strip().str.casefold()
        mais_comum = df.groupby(chave)[coluna].agg(lambda s: s.value_counts().index[0])
        antes = df[coluna].nunique()
        df[coluna] = chave.map(mais_comum)
        if verbose:
            print(f"{coluna}: {antes} grafias antes, {df[coluna].nunique()} depois.")
    return df

### Junção das edições e divisão em treino, validação e teste

Junta as bases limpas de Eficiência Acadêmica (edições 2021, 2022 e 2023,
geradas por `preprocessa_pnp.py`) e faz a divisão aleatória estratificada
70% treino / 20% validação / 10% teste, estratificando por edição e rótulo.

Saída: `ef_2021_2023.csv`, com a coluna `particao` indicando onde cada
matrícula caiu. Guardar a partição no arquivo garante que todo o notebook
(e quem reproduzir o trabalho) use exatamente a mesma divisão.

In [ ]:
if TEM_BRUTOS:
    import pandas as pd
    from sklearn.model_selection import train_test_split


    SEMENTE = 42
    EDICOES = [2021, 2022, 2023]
    PASTA_DADOS = "dados/processados/"
    ARQUIVOS = {ano: f"{PASTA_DADOS}ef_{ano}.csv" for ano in EDICOES}

#### 1. Carregar e conferir as edições

In [ ]:
if TEM_BRUTOS:
    bases = {ano: pd.read_csv(caminho) for ano, caminho in ARQUIVOS.items()}

    colunas = {ano: list(df.columns) for ano, df in bases.items()}
    referencia = colunas[EDICOES[0]]
    for ano, cols in colunas.items():
        if cols != referencia:
            faltam = set(referencia) - set(cols)
            sobram = set(cols) - set(referencia)
            raise ValueError(f"Edição {ano} tem colunas diferentes. Faltam: {faltam}. Sobram: {sobram}")
    print(f"As {len(EDICOES)} edições têm as mesmas {len(referencia)} colunas.")

    for ano, df in bases.items():
        anos_no_arquivo = sorted(df["ano_base"].unique())
        if anos_no_arquivo != [ano]:
            print(f"AVISO - ef_{ano}.csv traz ano_base {anos_no_arquivo}")

#### 2. Juntar

In [ ]:
if TEM_BRUTOS:
    df = pd.concat(bases.values(), ignore_index=True)

    # As categóricas viram texto em todas as edições, para não misturar tipos
    # (ex.: um campo lido como número numa edição e como texto em outra).
    numericas = ["carga_horaria", "ano_ingresso", "mes_ingresso",
                 "duracao_prevista_meses", "idade_ingresso", "ano_base", "evadiu",
                 "inscritos", "vagas", "inscritos_por_vaga"]
    numericas = [c for c in numericas if c in df.columns]
    categoricas = [c for c in df.columns if c not in numericas]
    df[categoricas] = df[categoricas].astype(str)

#### 2b. Padronizar nomes que mudaram entre edições

A edição 2023 traz o nome completo das instituições ("Instituto Federal do
Piauí"); as de 2021 e 2022 trazem a sigla ("IFPI"), e as escolas técnicas
vinculadas a universidades aparecem com siglas escritas de formas diferentes
("UFPI - CTF", "UFPI-CTF"). Sem padronizar, o modelo trataria a mesma
instituição como duas, uma para 2021–2022 e outra para 2023.
Padroniza também diferenças só de maiúsculas e minúsculas no nome do curso
e do campus ("Informática Para Internet" e "Informática para Internet").

In [ ]:
if TEM_BRUTOS:
    df = padroniza_nomes(df)

    resumo = (df.groupby("ano_base")["evadiu"]
                .agg(matriculas="size", taxa_evasao="mean")
                .assign(taxa_evasao=lambda t: (t["taxa_evasao"] * 100).round(1)))
    print(resumo.to_string())
    print(f"\nTotal: {len(df):,} matrículas, evasão de {df['evadiu'].mean():.1%}")

#### 3. Dividir 70 / 20 / 10

Estratifica pela combinação edição × rótulo, para que as três partes tenham
a mesma mistura de anos e a mesma taxa de evasão. Codificação, normalização
e imputação devem ser ajustadas **só no treino**, depois desta etapa.

In [ ]:
if TEM_BRUTOS:
    estrato = df["ano_base"].astype(str) + "_" + df["evadiu"].astype(str)
    idx_treino, idx_resto = train_test_split(
        df.index, test_size=0.30, stratify=estrato, random_state=SEMENTE)

    idx_validacao, idx_teste = train_test_split(
        idx_resto, test_size=1/3, stratify=estrato.loc[idx_resto], random_state=SEMENTE)

    df["particao"] = "treino"
    df.loc[idx_validacao, "particao"] = "validacao"
    df.loc[idx_teste, "particao"] = "teste"

    conferencia = (df.groupby("particao")
                     .agg(matriculas=("evadiu", "size"), taxa_evasao=("evadiu", "mean"))
                     .assign(proporcao=lambda t: (t["matriculas"] / len(df) * 100).round(1),
                             taxa_evasao=lambda t: (t["taxa_evasao"] * 100).round(1))
                     .loc[["treino", "validacao", "teste"]])
    print(conferencia.to_string())

    print("\nEdições por partição (%):")
    print((pd.crosstab(df["particao"], df["ano_base"], normalize="index") * 100)
          .round(1).loc[["treino", "validacao", "teste"]].to_string())

#### 4. Salvar

In [ ]:
if TEM_BRUTOS:
    df.to_csv(PASTA_DADOS + "ef_2021_2023.csv", index=False)
    print("Salvo em dados/processados/ef_2021_2023.csv")

    # Para usar depois:
    # df = pd.read_csv("dados/processados/ef_2021_2023.csv")
    # treino    = df[df["particao"] == "treino"]
    # validacao = df[df["particao"] == "validacao"]
    # teste     = df[df["particao"] == "teste"]

## 3. Análise exploratória

### Análise exploratória — evasão na Rede Federal do Nordeste

Usa **somente a partição de treino** de `ef_2021_2023.csv`. Validação e
teste ficam fechados até a avaliação, para que nada visto aqui influencie
decisões medidas neles.

Perguntas que esta análise responde:
1. Qual a qualidade dos atributos (ausentes, "Não declarada", categorias raras)?
2. Como a evasão varia com o curso, a oferta e o perfil do aluno?
3. Os atributos de perfil (sexo, cor/raça, renda) se confundem com o tipo de curso?
4. O IFPI difere do restante do Nordeste?

As figuras são salvas em `figuras/` para reuso no artigo.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMENTE = 42
PASTA_FIG = Path("figuras")
PASTA_FIG.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlesize": 12,
                     "axes.titleweight": "bold"})
COR = "#2b6cb0"      # destaque
CINZA = "#9aa5b1"

df_completo = pd.read_csv("dados/processados/ef_2021_2023.csv", dtype={"ano_base": int})
treino = df_completo[df_completo["particao"] == "treino"].copy()
TAXA_GERAL = treino["evadiu"].mean()
print(f"Treino: {len(treino):,} matrículas, evasão de {TAXA_GERAL:.1%}")

AUSENTES = {"Não informado", "Não declarada", "Não declarado", "nan", "Não Declarada"}
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}
treino["nivel"] = np.where(treino["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")

#### Funções auxiliares

In [ ]:
def milhar(n):
    """12345 -> '12.345'."""
    return f"{int(n):,}".replace(",", ".")


def wilson(k, n, z=1.96):
    """Intervalo de confiança de 95% (Wilson) para uma proporção."""
    p = k / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    meia = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - meia, centro + meia


def taxa_por(df, coluna, minimo=1):
    """Evasão por categoria, com n e IC 95%."""
    t = df.groupby(coluna)["evadiu"].agg(n="size", evadidos="sum")
    t = t[t["n"] >= minimo].copy()
    t["taxa"] = t["evadidos"] / t["n"]
    t["ic_inf"], t["ic_sup"] = wilson(t["evadidos"], t["n"])
    return t


def grafico_taxa(df, coluna, titulo, ordem=None, minimo=30, arquivo=None, destaque=None):
    """Barras horizontais de evasão por categoria, com IC 95% e a taxa geral como referência."""
    t = taxa_por(df, coluna, minimo)
    t = t.reindex([o for o in ordem if o in t.index]) if ordem else t.sort_values("taxa")
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(t) + 1.4))
    cores = [COR if (destaque and destaque(i)) else CINZA for i in t.index]
    ax.barh(t.index.astype(str), t["taxa"] * 100, color=cores,
            xerr=[(t["taxa"] - t["ic_inf"]) * 100, (t["ic_sup"] - t["taxa"]) * 100],
            error_kw={"ecolor": "#555", "capsize": 3, "lw": 1})
    ax.invert_yaxis()
    ax.axvline(TAXA_GERAL * 100, ls="--", lw=1, color="#555")
    for y, (taxa, sup, n) in enumerate(zip(t["taxa"], t["ic_sup"], t["n"])):
        ax.text(sup * 100 + 1.5, y, f"{taxa:.1%}  (n={milhar(n)})", va="center", fontsize=8)
    ax.set_xlim(0, 112)
    ax.set_xlabel(f"Taxa de evasão (%) · tracejado: geral {TAXA_GERAL:.1%} · barras: IC 95%")
    ax.set_title(titulo)
    fig.tight_layout()
    if arquivo:
        fig.savefig(PASTA_FIG / arquivo, bbox_inches="tight")
    plt.show()
    return t


def ordem_renda(categorias):
    """Ordena faixas de renda per capita pelo primeiro número; ausentes no fim."""
    def chave(c):
        if c in AUSENTES:
            return (1, 0.0)
        nums = re.findall(r"\d+(?:,\d+)?", c)
        return (0, float(nums[0].replace(",", ".")) if nums else 99.0)
    return sorted(categorias, key=chave)

#### 1. Qualidade dos dados

Proporção de valores ausentes ou não declarados por atributo e edição.
Atributos com muita ausência pedem cuidado: no modelo, "não declarado" pode
virar categoria própria; na análise de viés, os grupos ficam menos confiáveis.

In [ ]:
categoricas = [c for c in treino.columns if not pd.api.types.is_numeric_dtype(treino[c])
               and c not in ("particao", "nivel")]
ausencia = pd.DataFrame({
    ano: {c: g[c].isin(AUSENTES).mean() * 100 for c in categoricas}
    for ano, g in treino.groupby("ano_base")})
ausencia.loc["idade_ingresso"] = treino.groupby("ano_base")["idade_ingresso"].apply(
    lambda s: s.isna().mean() * 100)
ausencia = ausencia.round(2)
ausencia = ausencia[ausencia.max(axis=1) > 0].sort_values(ausencia.columns[-1], ascending=False)
print("Ausentes / não declarados (% das matrículas de treino):")
print(ausencia.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.45 * len(ausencia) + 1.4))
ausencia.plot.barh(ax=ax, color=["#c3cfe2", "#7f9cc7", COR], width=0.8)
ax.invert_yaxis()
ax.set_xlabel("% de matrículas sem a informação")
ax.set_title("Atributos com informação ausente ou não declarada")
ax.legend(title="Edição", frameon=False)
fig.tight_layout()
fig.savefig(PASTA_FIG / "01_ausentes.png", bbox_inches="tight")
plt.show()

##### Cardinalidade

Quantas categorias cada atributo tem e quantas são raras (menos de 30
matrículas no treino). Atributos com muitas categorias raras precisam de
agrupamento ou de uma codificação que não decore cada valor.

In [ ]:
cardinalidade = pd.DataFrame({
    "categorias": treino[categoricas].nunique(),
    "raras (<30)": [(treino[c].value_counts() < 30).sum() for c in categoricas],
    "% matrículas em raras": [
        treino[c].map(treino[c].value_counts()).lt(30).mean() * 100 for c in categoricas],
}).round(1).sort_values("categorias", ascending=False)
print(cardinalidade.to_string())

#### 2. Distribuição do rótulo

In [ ]:
por_edicao = taxa_por(treino, "ano_base")
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(por_edicao.index.astype(str), por_edicao["taxa"] * 100,
       color=[COR if a == por_edicao["taxa"].idxmax() else CINZA for a in por_edicao.index],
       yerr=[(por_edicao["taxa"] - por_edicao["ic_inf"]) * 100,
             (por_edicao["ic_sup"] - por_edicao["taxa"]) * 100], capsize=4)
for x, (taxa, n) in enumerate(zip(por_edicao["taxa"], por_edicao["n"])):
    ax.text(x, taxa * 100 + 4, f"{taxa:.1%}\nn={milhar(n)}", ha="center", fontsize=8)
ax.set_ylim(0, 80)
ax.set_xlabel("Edição da Eficiência Acadêmica")
ax.set_ylabel("Taxa de evasão (%)")
ax.set_title("Evasão por edição (maior em destaque)")
fig.tight_layout()
fig.savefig(PASTA_FIG / "02_evasao_por_edicao.png", bbox_inches="tight")
plt.show()

#### 3. Evasão por características do curso

In [ ]:
grafico_taxa(treino, "tipo_de_curso", "Evasão por tipo de curso (graduação em destaque)",
             arquivo="03_tipo_de_curso.png", destaque=lambda c: c in GRADUACAO)

In [ ]:
grafico_taxa(treino, "tipo_de_oferta", "Evasão por tipo de oferta",
             arquivo="04_tipo_de_oferta.png")

In [ ]:
grafico_taxa(treino, "turno", "Evasão por turno", arquivo="05_turno.png")

In [ ]:
grafico_taxa(treino, "modalidade_de_ensino", "Evasão por modalidade",
             arquivo="06_modalidade.png")

In [ ]:
grafico_taxa(treino, "eixo_tecnologico", "Evasão por eixo tecnológico",
             arquivo="07_eixo.png")

In [ ]:
# Duração prevista do curso, em faixas de 12 meses.
treino["faixa_duracao"] = pd.cut(treino["duracao_prevista_meses"],
                                 bins=[0, 12, 24, 36, 48, 60, 600],
                                 labels=["até 12", "13–24", "25–36", "37–48", "49–60", "mais de 60"])
grafico_taxa(treino, "faixa_duracao", "Evasão por duração prevista do curso (meses)",
             ordem=["até 12", "13–24", "25–36", "37–48", "49–60", "mais de 60"],
             arquivo="08_duracao.png")

#### 4. Evasão pelo perfil do aluno

Estes são os atributos da análise de viés. Os intervalos de confiança mostram
quais diferenças são estáveis e quais podem ser ruído de grupos pequenos.

In [ ]:
grafico_taxa(treino, "sexo", "Evasão por sexo", arquivo="09_sexo.png")

In [ ]:
grafico_taxa(treino, "cor_raca", "Evasão por cor/raça", arquivo="10_cor_raca.png",
             destaque=lambda c: c in AUSENTES)

In [ ]:
renda = taxa_por(treino, "renda_familiar")
grafico_taxa(treino, "renda_familiar", "Evasão por renda familiar per capita (salários mínimos)",
             ordem=ordem_renda(renda.index), arquivo="11_renda.png",
             destaque=lambda c: c in AUSENTES)

In [ ]:
treino["faixa_idade"] = pd.cut(treino["idade_ingresso"], bins=[0, 17, 20, 24, 29, 39, 120],
                               labels=["até 17", "18–20", "21–24", "25–29", "30–39", "40 ou mais"])
grafico_taxa(treino, "faixa_idade", "Evasão por idade no ingresso",
             ordem=["até 17", "18–20", "21–24", "25–29", "30–39", "40 ou mais"],
             arquivo="12_idade.png")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
bins = np.arange(10, 66, 2)
for rotulo, cor, nome in [(0, CINZA, "Concluiu"), (1, COR, "Evadiu")]:
    ax.hist(treino.loc[treino["evadiu"] == rotulo, "idade_ingresso"].dropna(),
            bins=bins, alpha=0.6, color=cor, label=nome, density=True)
ax.set_xlabel("Idade no ingresso (anos)")
ax.set_ylabel("Densidade")
ax.set_title("Distribuição da idade no ingresso por desfecho")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PASTA_FIG / "13_idade_hist.png", bbox_inches="tight")
plt.show()

#### 5. Perfil e tipo de curso se confundem?

Se um grupo se concentra na graduação, que evade mais, parte da diferença de
evasão desse grupo vem do curso e não do perfil. Comparar dentro de cada
nível (técnico e graduação) separa os dois efeitos.

In [ ]:
for atributo in ["sexo", "cor_raca", "renda_familiar"]:
    t = (treino.groupby([atributo, "nivel"])["evadiu"]
               .agg(n="size", taxa="mean").unstack("nivel"))
    t = t[(t["n"] >= 30).all(axis=1)]
    composicao = pd.crosstab(treino[atributo], treino["nivel"], normalize="index") * 100
    saida = pd.DataFrame({
        "% em graduação": composicao["Graduação"].round(1),
        "evasão técnico (%)": (t["taxa"]["Técnico"] * 100).round(1),
        "evasão graduação (%)": (t["taxa"]["Graduação"] * 100).round(1),
        "n técnico": t["n"]["Técnico"], "n graduação": t["n"]["Graduação"],
    }).dropna()
    if atributo == "renda_familiar":
        saida = saida.reindex(ordem_renda(saida.index))
    print(f"\n=== {atributo} ===")
    print(saida.to_string())

#### 6. Instituições e o IFPI

In [ ]:
inst = taxa_por(treino, "instituicao").sort_values("taxa", ascending=False)
fig, ax = plt.subplots(figsize=(8, 0.4 * len(inst) + 1.4))
eh_ifpi = inst.index == "Instituto Federal do Piauí"
ax.barh(inst.index, inst["taxa"] * 100, color=np.where(eh_ifpi, COR, CINZA),
        xerr=[(inst["taxa"] - inst["ic_inf"]) * 100, (inst["ic_sup"] - inst["taxa"]) * 100],
        error_kw={"ecolor": "#555", "capsize": 2, "lw": 1})
ax.axvline(TAXA_GERAL * 100, ls="--", lw=1, color="#555")
for y, (taxa, sup, n) in enumerate(zip(inst["taxa"], inst["ic_sup"], inst["n"])):
    ax.text(sup * 100 + 1.5, y, f"{taxa:.1%}  (n={milhar(n)})", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlim(0, 112)
ax.set_xlabel(f"Taxa de evasão (%) · tracejado: geral {TAXA_GERAL:.1%} · barras: IC 95%")
ax.set_title("Evasão por instituição (IFPI em destaque)")
fig.tight_layout()
fig.savefig(PASTA_FIG / "14_instituicoes.png", bbox_inches="tight")
plt.show()

In [ ]:
treino["grupo_ifpi"] = np.where(treino["instituicao"] == "Instituto Federal do Piauí",
                                "IFPI", "Resto do Nordeste")
comparacao = (treino.groupby(["nivel", "grupo_ifpi"])["evadiu"]
                    .agg(n="size", taxa="mean").unstack("grupo_ifpi"))
comparacao[("taxa", "IFPI")] *= 100
comparacao[("taxa", "Resto do Nordeste")] *= 100
print("IFPI x resto do Nordeste, por nível (taxa em %):")
print(comparacao.round(1).to_string())
print(f"\nMatrículas do IFPI no treino: {(treino['grupo_ifpi'] == 'IFPI').sum():,}")
print(f"Matrículas do IFPI no teste: "
      f"{(df_completo.query('particao == \"teste\"')['instituicao'] == 'Instituto Federal do Piauí').sum():,}")

#### 7. Resumo numérico para a documentação

In [ ]:
resumo = {
    "matrículas de treino": len(treino),
    "evasão geral (%)": round(TAXA_GERAL * 100, 1),
    "% técnico": round((treino["nivel"] == "Técnico").mean() * 100, 1),
    "% cor/raça não declarada": round(treino["cor_raca"].isin(AUSENTES).mean() * 100, 1),
    "% renda não declarada": round(treino["renda_familiar"].isin(AUSENTES).mean() * 100, 1),
    "instituições": treino["instituicao"].nunique(),
    "cursos distintos": treino["nome_de_curso"].nunique(),
    "campi": treino["unidade_de_ensino"].nunique(),
}
for k, v in resumo.items():
    print(f"{k}: {v:,}" if isinstance(v, int) else f"{k}: {v}")

## 4. Modelagem

### Modelagem — baselines, modelo principal e avaliação

Protocolo:
- **Treino** (70%): ajuste de todos os modelos e codificadores.
- **Validação** (20%): escolha de hiperparâmetros, calibração das
  probabilidades e escolha do limiar de decisão.
- **Teste** (10%): usado uma única vez, na seção 6, depois de tudo decidido.

Quatro conjuntos de atributos (decisões de 17/11 e 18/11):
- **sem indicadores:** dados do curso, da oferta, sexo e idade no ingresso;
- **só indicadores:** o anterior mais "não declarou renda" e "não declarou
  cor/raça". Esses registros são feitos no ingresso e explicam todo o ganho
  de renda e cor/raça (scripts/10_renda_raca.py);
- **principal:** só indicadores mais a cor/raça como entrada. Adotada porque
  reduz a subestimação do risco de estudantes indígenas sem perda relevante
  de AUC (scripts/11_cor_raca_entrada.py);
- **completo:** sem indicadores mais os valores de renda e cor/raça, com
  "não declarado" como categoria. Fica como comparação.

Saídas: `resultados/metricas_validacao.csv`, `resultados/metricas_teste.csv`,
`resultados/predicoes_teste.csv` (para a análise de viés),
`resultados/modelo_principal.joblib` (para a demonstração) e figuras em `figuras/`.

In [ ]:
import time
from itertools import product
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             brier_score_loss, f1_score, log_loss, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

SEMENTE = 42
np.random.seed(SEMENTE)
PASTA_FIG, PASTA_RES = Path("figuras"), Path("resultados")
PASTA_FIG.mkdir(exist_ok=True)
PASTA_RES.mkdir(exist_ok=True)
print(f"scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

COR, CINZA = "#2b6cb0", "#9aa5b1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})

#### 1. Dados e atributos

Ficam fora: `ano_base` (é a edição do arquivo, não uma característica do
aluno), `ano_ingresso` (na edição, é determinado pelo fim do ciclo e pela
duração, e não generaliza para turmas futuras), `regiao` (constante) e
`particao`.

In [ ]:
CAT_PRINCIPAL = ["instituicao", "unidade_de_ensino", "uf", "nome_de_curso", "tipo_de_curso",
                 "tipo_de_oferta", "modalidade_de_ensino", "eixo_tecnologico",
                 "subeixo_tecnologico", "turno", "fonte_de_financiamento", "sexo"]
NUM = ["carga_horaria", "duracao_prevista_meses", "mes_ingresso", "idade_ingresso"]
CAT_EXTRA = ["cor_raca", "renda_familiar"]
IND = ["renda_nao_declarada", "cor_nao_declarada"]
CONJUNTOS = {"sem indicadores": (CAT_PRINCIPAL, NUM),
             "só indicadores": (CAT_PRINCIPAL, NUM + IND),
             "principal": (CAT_PRINCIPAL + ["cor_raca"], NUM + IND),
             "completo": (CAT_PRINCIPAL + CAT_EXTRA, NUM)}
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}

df = pd.read_csv("dados/processados/ef_2021_2023.csv",
                 dtype={c: str for c in CAT_PRINCIPAL + CAT_EXTRA})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
treino = df[df["particao"] == "treino"].reset_index(drop=True)
validacao = df[df["particao"] == "validacao"].reset_index(drop=True)
teste = df[df["particao"] == "teste"].reset_index(drop=True)
y_tr, y_va, y_te = treino["evadiu"].values, validacao["evadiu"].values, teste["evadiu"].values
print({nome: len(p) for nome, p in [("treino", treino), ("validacao", validacao), ("teste", teste)]})

#### 2. Funções de avaliação

In [ ]:
def metricas(y, prob, limiar=0.5):
    """Métricas de ranqueamento, de probabilidade e de decisão (com o limiar dado)."""
    pred = (prob >= limiar).astype(int)
    return {
        "AUC": roc_auc_score(y, prob) if len(np.unique(prob)) > 1 else 0.5,
        "AP": average_precision_score(y, prob),
        "Brier": brier_score_loss(y, prob),
        "LogLoss": log_loss(y, np.clip(prob, 1e-6, 1 - 1e-6)),
        "F1": f1_score(y, pred, zero_division=0),
        "F1 macro": f1_score(y, pred, average="macro", zero_division=0),
        "Precisão": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "Acurácia": accuracy_score(y, pred),
        "Acurácia balanceada": balanced_accuracy_score(y, pred),
        "Limiar": limiar,
    }


def melhor_limiar(y, prob):
    """Limiar que maximiza a acurácia balanceada (escolhido na validação).

    Não se usa o F1 da classe 'evadiu' porque, com 58% de evasão, ele é
    maximizado prevendo evasão para todos, o que não serve para nada."""
    grade = np.linspace(0.05, 0.95, 91)
    bal = [balanced_accuracy_score(y, (prob >= t).astype(int)) for t in grade]
    return float(grade[int(np.argmax(bal))])


def auc_bootstrap(y, prob, n=500, semente=SEMENTE):
    """IC 95% da AUC por bootstrap."""
    rng = np.random.default_rng(semente)
    aucs = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) == 2:
            aucs.append(roc_auc_score(y[idx], prob[idx]))
    return np.percentile(aucs, [2.5, 97.5])

#### 3. Baselines

1. **Classe majoritária:** prevê para todos a taxa de evasão do treino.
2. **Taxa histórica do curso:** a evasão do mesmo curso no mesmo campus,
   no treino, suavizada em direção à taxa do tipo de curso (cursos pequenos
   não ficam com taxas extremas). Não usa nenhum dado do aluno; é o que uma
   coordenação já saberia sem modelo.
3. **Regressão logística** com one-hot nos categóricos.

In [ ]:
class TaxaHistoricaCurso:
    """Taxa de evasão por (campus, curso) no treino, suavizada pela taxa do tipo de curso."""

    def __init__(self, m=20):
        self.m = m

    def fit(self, X, y):
        d = X[["unidade_de_ensino", "nome_de_curso", "tipo_de_curso"]].assign(y=y)
        self.geral_ = d["y"].mean()
        self.tipo_ = d.groupby("tipo_de_curso")["y"].mean()
        g = d.groupby(["unidade_de_ensino", "nome_de_curso"]).agg(
            n=("y", "size"), soma=("y", "sum"), tipo=("tipo_de_curso", "first"))
        prior = g["tipo"].map(self.tipo_).fillna(self.geral_)
        self.curso_ = (g["soma"] + self.m * prior) / (g["n"] + self.m)
        return self

    def predict_proba(self, X):
        chave = pd.MultiIndex.from_frame(X[["unidade_de_ensino", "nome_de_curso"]])
        p = pd.Series(self.curso_.reindex(chave).values, index=X.index)
        p = p.fillna(X["tipo_de_curso"].map(self.tipo_)).fillna(self.geral_)
        return np.column_stack([1 - p.values, p.values])


def pipeline_logistica(cats, nums):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), cats),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("esc", StandardScaler())]), nums),
    ])
    return Pipeline([("pre", pre),
                     ("modelo", LogisticRegression(max_iter=3000, C=1.0))])


def pipeline_boosting(cats, nums, **params):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), cats),
        ("num", "passthrough", nums),
    ])
    modelo = HistGradientBoostingClassifier(
        categorical_features=list(range(len(cats))), max_iter=1000,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=30,
        random_state=SEMENTE, **params)
    return Pipeline([("pre", pre), ("modelo", modelo)])

In [ ]:
modelos = {}           # nome -> objeto treinado
prob_va = {}           # nome -> probabilidades na validação

modelos["Classe majoritária"] = None
prob_va["Classe majoritária"] = np.full(len(validacao), y_tr.mean())

modelos["Taxa histórica do curso"] = TaxaHistoricaCurso().fit(treino, y_tr)
prob_va["Taxa histórica do curso"] = modelos["Taxa histórica do curso"].predict_proba(validacao)[:, 1]

for conjunto, (cats, nums) in CONJUNTOS.items():
    nome = f"Regressão logística ({conjunto})"
    inicio = time.time()
    modelos[nome] = pipeline_logistica(cats, nums).fit(treino[cats + nums], y_tr)
    prob_va[nome] = modelos[nome].predict_proba(validacao[cats + nums])[:, 1]
    print(f"{nome}: {time.time() - inicio:.0f}s, AUC validação {roc_auc_score(y_va, prob_va[nome]):.4f}")

#### 4. Modelo principal: gradient boosting

Busca em grade pequena, escolhida pela AUC na validação. O próprio modelo
separa 10% do treino para parar cedo (early stopping), então a validação
externa não participa do ajuste dos pesos.

In [ ]:
GRADE = {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31, 63],
         "min_samples_leaf": [20, 100]}
combinacoes = [dict(zip(GRADE, v)) for v in product(*GRADE.values())]

busca = []
for conjunto, (cats, nums) in CONJUNTOS.items():
    for params in combinacoes:
        inicio = time.time()
        pipe = pipeline_boosting(cats, nums, l2_regularization=1.0, **params)
        pipe.fit(treino[cats + nums], y_tr)
        p = pipe.predict_proba(validacao[cats + nums])[:, 1]
        busca.append({"conjunto": conjunto, **params,
                      "iteracoes": pipe.named_steps["modelo"].n_iter_,
                      "AUC validação": roc_auc_score(y_va, p),
                      "segundos": round(time.time() - inicio, 1)})
        print(busca[-1])

busca = pd.DataFrame(busca).sort_values(["conjunto", "AUC validação"], ascending=[True, False])
busca.to_csv(PASTA_RES / "busca_hiperparametros.csv", index=False)
print(busca.groupby("conjunto").head(3).to_string(index=False))

In [ ]:
melhores = {}
for conjunto, (cats, nums) in CONJUNTOS.items():
    linha = busca[busca["conjunto"] == conjunto].iloc[0]
    params = {"learning_rate": float(linha["learning_rate"]),
              "max_leaf_nodes": int(linha["max_leaf_nodes"]),
              "min_samples_leaf": int(linha["min_samples_leaf"])}
    melhores[conjunto] = params
    nome = f"Gradient boosting ({conjunto})"
    modelos[nome] = pipeline_boosting(cats, nums, l2_regularization=1.0, **params).fit(
        treino[cats + nums], y_tr)
    prob_va[nome] = modelos[nome].predict_proba(validacao[cats + nums])[:, 1]
    print(f"{nome}: {params}, AUC validação {roc_auc_score(y_va, prob_va[nome]):.4f}")

#### 5. Calibração e limiar (na validação)

A saída do sistema é uma probabilidade, então ela precisa ser confiável:
se o modelo diz 70%, cerca de 70% daqueles alunos devem evadir. A
calibração isotônica é ajustada nas previsões da validação; por ser
monótona, não altera a AUC. O limiar de decisão é o que maximiza a acurácia
balanceada (média do recall das duas classes) na validação, e é aplicado
sem mudança no teste.

In [ ]:
def colunas_de(nome):
    if nome in ("Classe majoritária", "Taxa histórica do curso"):
        return None
    for conjunto, (cats, nums) in CONJUNTOS.items():
        if f"({conjunto})" in nome:
            return cats + nums
    raise KeyError(nome)


calibradores, limiares = {}, {}
for nome, p in list(prob_va.items()):
    if nome.startswith("Gradient boosting") or nome.startswith("Regressão"):
        calibradores[nome] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y_va)
        prob_va[nome + " · calibrado"] = calibradores[nome].predict(p)
for nome, p in prob_va.items():
    limiares[nome] = melhor_limiar(y_va, p)

tabela_va = pd.DataFrame({nome: metricas(y_va, p, limiares[nome]) for nome, p in prob_va.items()}).T
tabela_va.to_csv(PASTA_RES / "metricas_validacao.csv")
print(tabela_va.round(4).sort_values("AUC").to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for nome, cor, estilo in [("Gradient boosting (principal)", CINZA, "--"),
                          ("Gradient boosting (principal) · calibrado", COR, "-")]:
    frac, media = calibration_curve(y_va, prob_va[nome], n_bins=10, strategy="quantile")
    ax.plot(media, frac, estilo, marker="o", color=cor,
            label=nome.replace("Gradient boosting (principal)", "Antes").replace("Antes · calibrado", "Depois da calibração"))
ax.plot([0, 1], [0, 1], ":", color="#555", label="Calibração perfeita")
ax.set_xlabel("Probabilidade prevista de evasão")
ax.set_ylabel("Proporção que evadiu")
ax.set_title("Calibração do modelo principal (validação)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PASTA_FIG / "20_calibracao_validacao.png", bbox_inches="tight")
plt.show()

##### Importância dos atributos (validação)

Importância por permutação: quanto a AUC cai quando os valores de um
atributo são embaralhados. Calculada numa amostra de 20 mil matrículas da
validação para economizar tempo.

In [ ]:
amostra = validacao.sample(min(20000, len(validacao)), random_state=SEMENTE)
cols = CONJUNTOS["principal"][0] + CONJUNTOS["principal"][1]
imp = permutation_importance(modelos["Gradient boosting (principal)"], amostra[cols],
                             amostra["evadiu"], scoring="roc_auc", n_repeats=5,
                             random_state=SEMENTE, n_jobs=-1)
importancia = pd.DataFrame({"atributo": cols, "queda_auc": imp.importances_mean,
                            "dp": imp.importances_std}).sort_values("queda_auc")
importancia.to_csv(PASTA_RES / "importancia_permutacao.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 0.35 * len(importancia) + 1.2))
ax.barh(importancia["atributo"], importancia["queda_auc"], xerr=importancia["dp"],
        color=[COR if v == importancia["queda_auc"].max() else CINZA for v in importancia["queda_auc"]])
ax.set_xlabel("Queda da AUC ao embaralhar o atributo")
ax.set_title("O que o modelo principal usa")
fig.tight_layout()
fig.savefig(PASTA_FIG / "21_importancia.png", bbox_inches="tight")
plt.show()

#### 6. Avaliação final no teste

Executada uma vez, com tudo o que foi decidido acima congelado: modelos,
hiperparâmetros, calibradores e limiares.

In [ ]:
prob_te = {}
prob_te["Classe majoritária"] = np.full(len(teste), y_tr.mean())
prob_te["Taxa histórica do curso"] = modelos["Taxa histórica do curso"].predict_proba(teste)[:, 1]
for nome, modelo in modelos.items():
    cols = colunas_de(nome)
    if cols is None:
        continue
    prob_te[nome] = modelo.predict_proba(teste[cols])[:, 1]
    if nome in calibradores:
        prob_te[nome + " · calibrado"] = calibradores[nome].predict(prob_te[nome])

linhas = {}
for nome, p in prob_te.items():
    m = metricas(y_te, p, limiares[nome])
    constante = len(np.unique(p)) == 1
    m["AUC IC inf"], m["AUC IC sup"] = (m["AUC"], m["AUC"]) if constante else auc_bootstrap(y_te, p)
    linhas[nome] = m
tabela_te = pd.DataFrame(linhas).T
tabela_te.to_csv(PASTA_RES / "metricas_teste.csv")
print(tabela_te.round(4).sort_values("AUC").to_string())

In [ ]:
ordem = ["Classe majoritária", "Taxa histórica do curso", "Regressão logística (principal)",
         "Gradient boosting (sem indicadores) · calibrado", "Gradient boosting (só indicadores) · calibrado",
         "Gradient boosting (principal) · calibrado", "Gradient boosting (completo) · calibrado"]
t = tabela_te.loc[ordem]
fig, ax = plt.subplots(figsize=(8, 4.4))
cores = [COR if n == "Gradient boosting (principal) · calibrado" else CINZA for n in ordem]
ax.barh(range(len(t)), t["AUC"], color=cores,
        xerr=[np.clip(t["AUC"] - t["AUC IC inf"], 0, None), np.clip(t["AUC IC sup"] - t["AUC"], 0, None)],
        error_kw={"ecolor": "#555", "capsize": 3})
ax.set_yticks(range(len(t)), [n.replace(" · calibrado", "") for n in ordem])
ax.invert_yaxis()
for y, (auc, sup) in enumerate(zip(t["AUC"], t["AUC IC sup"])):
    ax.text(sup + 0.005, y, f"{auc:.3f}", va="center", fontsize=8)
ax.axvline(0.5, ls=":", color="#555")
ax.set_xlim(0.45, 1.0)
ax.set_xlabel("AUC no teste · barras: IC 95% por bootstrap · pontilhado: acaso")
ax.set_title("Comparação dos modelos no teste")
fig.tight_layout()
fig.savefig(PASTA_FIG / "22_comparacao_auc_teste.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for nome, cor in [("Taxa histórica do curso", CINZA), ("Regressão logística (principal)", "#7f9cc7"),
                  ("Gradient boosting (principal) · calibrado", COR)]:
    fpr, tpr, _ = roc_curve(y_te, prob_te[nome])
    ax.plot(fpr, tpr, color=cor, label=f"{nome.replace(' · calibrado', '')} ({tabela_te.loc[nome, 'AUC']:.3f})")
ax.plot([0, 1], [0, 1], ":", color="#555")
ax.set_xlabel("Taxa de falsos positivos")
ax.set_ylabel("Taxa de verdadeiros positivos (recall)")
ax.set_title("Curvas ROC no teste")
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(PASTA_FIG / "23_roc_teste.png", bbox_inches="tight")
plt.show()

#### 7. Salvar para as próximas etapas

In [ ]:
saida = teste[["instituicao", "uf", "tipo_de_curso", "sexo", "cor_raca", "renda_familiar",
               "idade_ingresso", "ano_base", "evadiu"]].copy()
saida["prob_principal"] = prob_te["Gradient boosting (principal) · calibrado"]
saida["prob_completo"] = prob_te["Gradient boosting (completo) · calibrado"]
saida["prob_sem_indicadores"] = prob_te["Gradient boosting (sem indicadores) · calibrado"]
saida["prob_so_indicadores"] = prob_te["Gradient boosting (só indicadores) · calibrado"]
saida["prob_taxa_curso"] = prob_te["Taxa histórica do curso"]
saida["limiar_principal"] = limiares["Gradient boosting (principal) · calibrado"]
saida.to_csv(PASTA_RES / "predicoes_teste.csv", index=False)

joblib.dump({"modelo": modelos["Gradient boosting (principal)"],
             "calibrador": calibradores["Gradient boosting (principal)"],
             "limiar": limiares["Gradient boosting (principal) · calibrado"],
             "colunas": CONJUNTOS["principal"][0] + CONJUNTOS["principal"][1],
             "hiperparametros": melhores["principal"]},
            PASTA_RES / "modelo_principal.joblib")
print("Salvos: predicoes_teste.csv e modelo_principal.joblib em resultados/")

## 5. Análise de viés

### Análise de viés do modelo principal

Usa as previsões do **teste** salvas por `modelagem.py`
(`resultados/predicoes_teste.csv`). Nenhum modelo é ajustado aqui.

Para cada grupo de sexo, cor/raça, renda e idade, compara:
- **Recall** (taxa de verdadeiros positivos): dos alunos do grupo que
  evadiram, quantos o modelo marcou como risco. Recall menor = o grupo fica
  mais sem apoio.
- **Taxa de falsos positivos**: dos que concluíram, quantos foram marcados
  como risco. Taxa maior = o grupo é mais rotulado sem motivo.
- **Taxa de alerta**: proporção do grupo marcada como risco.
- **Calibração**: probabilidade média prevista contra a evasão observada.
- **AUC dentro do grupo**: se o modelo ordena bem os alunos daquele grupo.

Como o tipo de curso pesa muito na evasão, tudo é repetido dentro do
técnico e da graduação. Grupos com menos de 30 evadidos ou 30 concluintes
não têm recall ou falsos positivos calculados: o intervalo seria largo
demais para dizer algo.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SEMENTE = 42
MIN_CLASSE = 30
PASTA_FIG, PASTA_RES = Path("figuras"), Path("resultados")
PASTA_FIG.mkdir(exist_ok=True)
COR, CINZA, COR2 = "#2b6cb0", "#9aa5b1", "#dd6b20"
COR_NIVEL = {"Técnico": COR, "Graduação": COR2}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})

AUSENTES = {"Não informado", "Não declarada", "Não declarado", "nan", "Não Declarada"}
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}

p = pd.read_csv(PASTA_RES / "predicoes_teste.csv")
LIMIAR = float(p["limiar_principal"].iloc[0])
p["alerta"] = (p["prob_principal"] >= LIMIAR).astype(int)
p["nivel"] = np.where(p["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
p["faixa_idade"] = pd.cut(p["idade_ingresso"], bins=[0, 17, 20, 24, 29, 120],
                          labels=["até 17", "18–20", "21–24", "25–29", "30 ou mais"]).astype(str)
p["grupo_ifpi"] = np.where(p["instituicao"] == "Instituto Federal do Piauí", "IFPI",
                           "Resto do Nordeste")
print(f"Teste: {len(p):,} matrículas · limiar {LIMIAR:.2f} · evasão real {p['evadiu'].mean():.1%} "
      f"· taxa de alerta {p['alerta'].mean():.1%}")

#### Funções

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    q = k / n
    centro = (q + z**2 / (2 * n)) / (1 + z**2 / n)
    meia = z * np.sqrt(q * (1 - q) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - meia, centro + meia


def auc_ic(y, prob, n=300, semente=SEMENTE):
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(semente)
    aucs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            aucs.append(roc_auc_score(y[i], prob[i]))
    return roc_auc_score(y, prob), *np.percentile(aucs, [2.5, 97.5])


def metricas_por_grupo(df, coluna, prob="prob_principal"):
    linhas = []
    for grupo, g in df.groupby(coluna):
        pos, neg = g[g["evadiu"] == 1], g[g["evadiu"] == 0]
        l = {"grupo": grupo, "n": len(g), "evadidos": len(pos), "concluintes": len(neg),
             "evasão real": g["evadiu"].mean(), "prob. média": g[prob].mean(),
             "taxa de alerta": g["alerta"].mean()}
        l["erro de calibração"] = l["prob. média"] - l["evasão real"]
        if len(pos) >= MIN_CLASSE and len(neg) >= MIN_CLASSE:
            l["recall"] = pos["alerta"].mean()
            l["recall ic inf"], l["recall ic sup"] = wilson(pos["alerta"].sum(), len(pos))
            l["falsos positivos"] = neg["alerta"].mean()
            l["fp ic inf"], l["fp ic sup"] = wilson(neg["alerta"].sum(), len(neg))
            l["AUC"], l["AUC ic inf"], l["AUC ic sup"] = auc_ic(g["evadiu"].values, g[prob].values)
        linhas.append(l)
    colunas = ["n", "evadidos", "concluintes", "evasão real", "prob. média", "taxa de alerta",
               "erro de calibração", "recall", "recall ic inf", "recall ic sup",
               "falsos positivos", "fp ic inf", "fp ic sup", "AUC", "AUC ic inf", "AUC ic sup"]
    return pd.DataFrame(linhas).set_index("grupo").reindex(columns=colunas)


def ordem_renda(categorias):
    def chave(c):
        if c in AUSENTES:
            return (1, 0.0)
        nums = re.findall(r"\d+(?:,\d+)?", c)
        return (0, float(nums[0].replace(",", ".")) if nums else 99.0)
    return sorted(categorias, key=chave)


ORDENS = {
    "faixa_idade": ["até 17", "18–20", "21–24", "25–29", "30 ou mais"],
    "renda_familiar": None,   # calculada a partir dos dados
}


def ordenar(t, coluna):
    if coluna == "renda_familiar":
        return t.reindex(ordem_renda(t.index))
    if coluna in ORDENS and ORDENS[coluna]:
        return t.reindex([o for o in ORDENS[coluna] if o in t.index])
    return t.sort_values("n", ascending=False)


def resumo_disparidade(t, nome):
    """Maior diferença entre grupos com recall calculado."""
    v = t.dropna(subset=["recall"])
    v = v[~v.index.isin(AUSENTES)]
    if len(v) < 2:
        return None
    return {"atributo": nome, "grupos comparados": len(v),
            "diferença de recall": v["recall"].max() - v["recall"].min(),
            "menor recall": v["recall"].idxmin(),
            "diferença de falsos positivos": v["falsos positivos"].max() - v["falsos positivos"].min(),
            "maior taxa de falsos positivos": v["falsos positivos"].idxmax(),
            "maior erro de calibração (pontos)": (v["erro de calibração"].abs().max() * 100)}


def grafico_grupos(tabelas, coluna, titulo, arquivo):
    """Recall e falsos positivos por grupo, um ponto por nível de curso, com IC 95%."""
    grupos = list(dict.fromkeys(g for t in tabelas.values() for g in t.dropna(subset=["recall"]).index))
    if not grupos:
        return
    fig, eixos = plt.subplots(1, 2, figsize=(10, 0.5 * len(grupos) + 1.8), sharey=True)
    for ax, (met, inf, sup, rotulo) in zip(eixos, [
            ("recall", "recall ic inf", "recall ic sup", "Recall: evadidos marcados como risco (%)"),
            ("falsos positivos", "fp ic inf", "fp ic sup", "Concluintes marcados como risco (%)")]):
        for k, (nivel, t) in enumerate(tabelas.items()):
            t = t.reindex(grupos)
            y = np.arange(len(grupos)) + (k - 0.5) * 0.25
            ok = t[met].notna().values
            ax.errorbar(t[met].values[ok] * 100, y[ok],
                        xerr=[(t[met] - t[inf]).values[ok] * 100, (t[sup] - t[met]).values[ok] * 100],
                        fmt="o", color=COR_NIVEL[nivel], label=nivel, capsize=3, ms=5)
        ax.set_yticks(range(len(grupos)), grupos)
        ax.invert_yaxis()
        ax.set_xlim(0, 100)
        ax.set_xlabel(rotulo)
        ax.grid(axis="x", alpha=0.3)
    alcas, rotulos = eixos[0].get_legend_handles_labels()
    fig.legend(alcas, rotulos, frameon=False, loc="upper right", ncol=2)
    fig.suptitle(titulo, fontweight="bold", x=0.02, ha="left")
    fig.tight_layout()
    fig.savefig(PASTA_FIG / arquivo, bbox_inches="tight")
    plt.show()

#### 1. Visão geral por nível de curso

In [ ]:
geral = metricas_por_grupo(p, "nivel")
print(geral.round(3).T.to_string())

#### 2. Grupos de perfil

Para cada atributo: a tabela de todos os alunos, depois técnico e graduação
separados, e o gráfico de recall e falsos positivos.

In [ ]:
ATRIBUTOS = {"sexo": "Sexo", "cor_raca": "Cor/raça", "renda_familiar": "Renda familiar per capita",
             "faixa_idade": "Idade no ingresso"}
COLS_TABELA = ["n", "evasão real", "prob. média", "erro de calibração", "taxa de alerta",
               "recall", "falsos positivos", "AUC"]

tabelas_salvas, disparidades = [], []
for i, (coluna, nome) in enumerate(ATRIBUTOS.items(), start=30):
    todos = ordenar(metricas_por_grupo(p, coluna), coluna)
    por_nivel = {nivel: ordenar(metricas_por_grupo(g, coluna), coluna)
                 for nivel, g in p.groupby("nivel")}
    print(f"\n========== {nome} ==========")
    print("Todos:")
    print(todos[COLS_TABELA].round(3).to_string())
    for nivel, t in por_nivel.items():
        print(f"\n{nivel}:")
        print(t[COLS_TABELA].round(3).to_string())
        d = resumo_disparidade(t, f"{nome} · {nivel}")
        if d:
            disparidades.append(d)
    d = resumo_disparidade(todos, f"{nome} · todos")
    if d:
        disparidades.append(d)
    for escopo, t in [("todos", todos), *por_nivel.items()]:
        tabelas_salvas.append(t.assign(atributo=coluna, escopo=escopo))
    grafico_grupos(por_nivel, coluna, f"{nome}: recall e falsos positivos por nível de curso",
                   f"{i}_vies_{coluna}.png")

pd.concat(tabelas_salvas).to_csv(PASTA_RES / "vies_por_grupo.csv")

#### 3. Calibração por grupo

Pontos sobre a diagonal = a probabilidade média prevista para o grupo bate
com a evasão que ele teve. Abaixo da diagonal = o modelo superestima o
risco do grupo; acima = subestima.

In [ ]:
pontos = pd.concat([
    ordenar(metricas_por_grupo(g, c), c).assign(atributo=ATRIBUTOS[c], nivel=nivel)
    for c in ATRIBUTOS for nivel, g in p.groupby("nivel")])
pontos = pontos[pontos["n"] >= 100]
fig, ax = plt.subplots(figsize=(6, 6))
for nivel, cor in COR_NIVEL.items():
    s = pontos[pontos["nivel"] == nivel]
    ax.scatter(s["prob. média"] * 100, s["evasão real"] * 100, s=np.clip(s["n"] / 15, 15, 200),
               color=cor, alpha=0.7, label=nivel)
pior = pontos.loc[pontos["erro de calibração"].abs().nlargest(4).index.unique()]
for g, l in pior.iterrows():
    ax.annotate(f"{l['atributo']}: {g} ({l['nivel'][:4]}.)", (l["prob. média"] * 100, l["evasão real"] * 100),
                textcoords="offset points", xytext=(6, -10), fontsize=8)
ax.plot([0, 100], [0, 100], ":", color="#555")
ax.set_xlim(20, 100)
ax.set_ylim(20, 100)
ax.set_xlabel("Probabilidade média prevista (%)")
ax.set_ylabel("Evasão observada (%)")
ax.set_title("Calibração por grupo (tamanho = n; grupos com n ≥ 100)")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(PASTA_FIG / "34_calibracao_grupos.png", bbox_inches="tight")
plt.show()

#### 4. IFPI

In [ ]:
ifpi = pd.concat({nivel: metricas_por_grupo(g, "grupo_ifpi") for nivel, g in p.groupby("nivel")})
print(ifpi[COLS_TABELA + ["recall ic inf", "recall ic sup"]].round(3).to_string())

#### 5. Resumo das disparidades

Para cada atributo e escopo, a maior diferença de recall e de falsos
positivos entre grupos declarados (o "não declarado" fica de fora da
comparação, porque não é um grupo de pessoas e sim uma falta de registro).

In [ ]:
resumo = pd.DataFrame(disparidades)
for c in ["diferença de recall", "diferença de falsos positivos"]:
    resumo[c] = (resumo[c] * 100).round(1)
resumo["maior erro de calibração (pontos)"] = resumo["maior erro de calibração (pontos)"].round(1)
resumo.to_csv(PASTA_RES / "vies_resumo.csv", index=False)
print(resumo.to_string(index=False))

## 6. Experimentos de melhoria

### Experimentos de melhoria do modelo principal

Três perguntas, todas respondidas na **validação**; o teste é usado uma
única vez, na seção 5, só para o que for escolhido aqui.

1. **Novos atributos** ajudam?
   - *idade relativa*: idade no ingresso menos a idade mediana de quem entra
     no mesmo curso, campus e tipo de oferta (calculada só no treino);
   - *demanda*: inscritos por vaga do curso no ano-base (precisa das colunas
     `inscritos`, `vagas` e `inscritos_por_vaga`, geradas pela versão atual
     de `preprocessa_pnp.py`).
2. **Limiar por nível de curso** (técnico e graduação) melhora as decisões?
3. **Curva de aprendizado:** mais dados ajudariam?

Hiperparâmetros: os do modelo principal escolhidos em `modelagem.py`
(a busca mostrou que eles quase não importam).

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEMENTE = 42
PASTA_FIG, PASTA_RES = Path("figuras"), Path("resultados")
COR, CINZA, COR2 = "#2b6cb0", "#9aa5b1", "#dd6b20"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})

CAT = ["instituicao", "unidade_de_ensino", "uf", "nome_de_curso", "tipo_de_curso",
       "tipo_de_oferta", "modalidade_de_ensino", "eixo_tecnologico", "subeixo_tecnologico",
       "turno", "fonte_de_financiamento", "sexo", "cor_raca"]
# Numéricos do modelo principal, incluindo os indicadores de não declaração (decisão de 18/11).
NUM = ["carga_horaria", "duracao_prevista_meses", "mes_ingresso", "idade_ingresso",
       "renda_nao_declarada", "cor_nao_declarada"]
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}

anterior = joblib.load(PASTA_RES / "modelo_principal.joblib")
PARAMS = anterior["hiperparametros"]
print("Hiperparâmetros do modelo principal:", PARAMS)

df = pd.read_csv("dados/processados/ef_2021_2023.csv", dtype={c: str for c in CAT})
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
treino = df[df["particao"] == "treino"].reset_index(drop=True)
validacao = df[df["particao"] == "validacao"].reset_index(drop=True)
teste = df[df["particao"] == "teste"].reset_index(drop=True)
for d in (treino, validacao, teste):
    d["nivel"] = np.where(d["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
y_tr, y_va, y_te = treino["evadiu"].values, validacao["evadiu"].values, teste["evadiu"].values
TEM_DEMANDA = "inscritos_por_vaga" in df.columns
print(f"Colunas de demanda disponíveis: {TEM_DEMANDA}")

#### 1. Novos atributos

##### 1.1 Idade relativa ao curso

A mediana vem só do treino. Para cursos que não aparecem no treino, cai
para a mediana do nome de curso e, por fim, do tipo de curso.

In [ ]:
NIVEIS_MEDIANA = [["unidade_de_ensino", "nome_de_curso", "tipo_de_oferta"],
                  ["nome_de_curso", "tipo_de_oferta"],
                  ["tipo_de_curso", "tipo_de_oferta"],
                  ["tipo_de_curso"]]
medianas = [treino.groupby(chaves)["idade_ingresso"].median().rename("m") for chaves in NIVEIS_MEDIANA]


def idade_relativa(d):
    tipica = pd.Series(np.nan, index=d.index)
    for chaves, m in zip(NIVEIS_MEDIANA, medianas):
        tipica = tipica.fillna(d[chaves].join(m, on=chaves)["m"])
    return d["idade_ingresso"] - tipica


for d in (treino, validacao, teste):
    d["idade_relativa"] = idade_relativa(d)
print(treino.groupby("nivel")["idade_relativa"].describe().round(1).to_string())

##### 1.2 Demanda do curso: diagnóstico antes de usar

Inscritos e vagas são do ano-base da edição, não do ano de ingresso do
aluno. Antes de usar, confere-se quanto falta e se o valor é estável dentro
de um mesmo curso e campus.

In [ ]:
if TEM_DEMANDA:
    for d in (treino, validacao, teste):
        d["log_inscritos_por_vaga"] = np.log1p(d["inscritos_por_vaga"])
    print(f"Sem relação inscritos/vaga (vagas = 0 ou ausente): "
          f"{treino['inscritos_por_vaga'].isna().mean():.1%} do treino")
    print(treino["inscritos_por_vaga"].describe().round(2).to_string())
    variacao = (treino.groupby(["ano_base", "unidade_de_ensino", "nome_de_curso", "turno"])
                      ["inscritos_por_vaga"].nunique())
    print(f"Grupos (edição, campus, curso, turno) com mais de um valor: {(variacao > 1).mean():.1%}")
    faixa = pd.qcut(treino["inscritos_por_vaga"], 5, duplicates="drop")
    print("\nEvasão por quintil de inscritos por vaga (treino):")
    print(treino.groupby(faixa, observed=True)["evadiu"].agg(["size", "mean"]).round(3).to_string())
    print(f"Evasão quando a relação está ausente: "
          f"{treino.loc[treino['inscritos_por_vaga'].isna(), 'evadiu'].mean():.3f}")

##### 1.3 Comparação na validação

In [ ]:
def pipeline(nums):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), CAT),
        ("num", "passthrough", nums)])
    modelo = HistGradientBoostingClassifier(
        categorical_features=list(range(len(CAT))), max_iter=1000, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=30, l2_regularization=1.0,
        random_state=SEMENTE, **PARAMS)
    return Pipeline([("pre", pre), ("modelo", modelo)])


VARIANTES = {"atual": NUM, "+ idade relativa": NUM + ["idade_relativa"]}
if TEM_DEMANDA:
    VARIANTES["+ demanda"] = NUM + ["log_inscritos_por_vaga"]
    VARIANTES["+ idade relativa + demanda"] = NUM + ["idade_relativa", "log_inscritos_por_vaga"]

modelos, prob_va, linhas = {}, {}, []
for nome, nums in VARIANTES.items():
    modelos[nome] = pipeline(nums).fit(treino[CAT + nums], y_tr)
    prob_va[nome] = modelos[nome].predict_proba(validacao[CAT + nums])[:, 1]
    jovens = (validacao["idade_ingresso"] <= 17).values
    linhas.append({"variante": nome,
                   "AUC validação": roc_auc_score(y_va, prob_va[nome]),
                   "AUC até 17 anos": roc_auc_score(y_va[jovens], prob_va[nome][jovens]),
                   "Brier": brier_score_loss(y_va, prob_va[nome])})
comparacao = pd.DataFrame(linhas).set_index("variante")
comparacao["ganho AUC"] = comparacao["AUC validação"] - comparacao.loc["atual", "AUC validação"]
print(comparacao.round(4).to_string())

Escolha: a variante de maior AUC na validação, desde que o ganho sobre a
atual seja de pelo menos 0,002 (abaixo disso é ruído, como mostrou a busca
de hiperparâmetros). Se nenhuma passar, o modelo atual fica.

In [ ]:
melhor = comparacao["AUC validação"].idxmax()
ESCOLHIDA = melhor if comparacao.loc[melhor, "ganho AUC"] >= 0.002 else "atual"
print(f"Variante escolhida: {ESCOLHIDA}")

#### 2. Limiar por nível de curso

Hoje há um limiar único. Aqui se escolhe, na validação, um limiar para o
técnico e outro para a graduação, cada um maximizando a acurácia
balanceada dentro do seu nível. As probabilidades são calibradas antes
(isotônica na validação), como no modelo principal.

In [ ]:
calibrador = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(prob_va[ESCOLHIDA], y_va)
pcal_va = calibrador.predict(prob_va[ESCOLHIDA])
GRADE = np.linspace(0.05, 0.95, 91)


def limiar_bal(y, p):
    return float(GRADE[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in GRADE]))])


LIMIAR_UNICO = limiar_bal(y_va, pcal_va)
LIMIARES_NIVEL = {nivel: limiar_bal(y_va[(validacao["nivel"] == nivel).values],
                                    pcal_va[(validacao["nivel"] == nivel).values])
                  for nivel in ["Técnico", "Graduação"]}
print(f"Limiar único: {LIMIAR_UNICO:.2f} · por nível: {LIMIARES_NIVEL}")


def decisoes(d, pcal, regra):
    lim = np.full(len(d), LIMIAR_UNICO) if regra == "único" else d["nivel"].map(LIMIARES_NIVEL).values
    return (pcal >= lim).astype(int)


def tabela_decisoes(d, y, pcal):
    linhas = []
    grupos = {"Todos": np.ones(len(d), bool),
              "Técnico": (d["nivel"] == "Técnico").values,
              "Graduação": (d["nivel"] == "Graduação").values,
              "Até 17 anos": (d["idade_ingresso"] <= 17).values}
    for regra in ["único", "por nível"]:
        alerta = decisoes(d, pcal, regra)
        for g, m in grupos.items():
            yy, aa = y[m], alerta[m]
            linhas.append({"regra": regra, "grupo": g, "n": int(m.sum()),
                           "taxa de alerta": aa.mean(),
                           "recall": aa[yy == 1].mean(),
                           "falsos positivos": aa[yy == 0].mean(),
                           "acurácia balanceada": balanced_accuracy_score(yy, aa)})
    return pd.DataFrame(linhas).set_index(["grupo", "regra"]).sort_index()


print(tabela_decisoes(validacao, y_va, pcal_va).round(3).to_string())

#### 3. Curva de aprendizado

Treina a variante escolhida com frações do treino (amostras estratificadas)
e mede a AUC na validação. Se a curva ainda sobe no fim, mais dados
ajudariam; se achatou, o limite é a informação disponível.

In [ ]:
FRACOES = [0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
SEMENTES = [SEMENTE, SEMENTE + 1, SEMENTE + 2]
nums = VARIANTES[ESCOLHIDA]
curva = []
for f in FRACOES:
    for s in (SEMENTES if f < 1 else [SEMENTE]):
        if f < 1:
            sub, _ = train_test_split(treino, train_size=f, stratify=treino["evadiu"], random_state=s)
        else:
            sub = treino
        m = pipeline(nums).fit(sub[CAT + nums], sub["evadiu"])
        curva.append({"fração": f, "n": len(sub), "semente": s,
                      "AUC": roc_auc_score(y_va, m.predict_proba(validacao[CAT + nums])[:, 1])})
curva = pd.DataFrame(curva)
resumo_curva = curva.groupby(["fração", "n"])["AUC"].agg(["mean", "min", "max"]).reset_index()
print(resumo_curva.round(4).to_string(index=False))
curva.to_csv(PASTA_RES / "curva_aprendizado.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.fill_between(resumo_curva["n"], resumo_curva["min"], resumo_curva["max"], color=COR, alpha=0.15)
ax.plot(resumo_curva["n"], resumo_curva["mean"], "o-", color=COR)
for _, l in resumo_curva.iterrows():
    ax.annotate(f"{l['mean']:.3f}", (l["n"], l["mean"]), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Matrículas de treino (escala log)")
ax.set_ylabel("AUC na validação")
ax.set_title("Curva de aprendizado do modelo principal")
fig.tight_layout()
fig.savefig(PASTA_FIG / "40_curva_aprendizado.png", bbox_inches="tight")
plt.show()

#### 4. Resumo das escolhas

Atributos: a variante escolhida na seção 1.3. Limiar: por nível se ele
aumentar a acurácia balanceada média dos dois níveis na validação; senão,
o único.

In [ ]:
dec_va = tabela_decisoes(validacao, y_va, pcal_va)
bal_niveis = {regra: dec_va.loc[[("Técnico", regra), ("Graduação", regra)], "acurácia balanceada"].mean()
              for regra in ["único", "por nível"]}
REGRA = "por nível" if bal_niveis["por nível"] > bal_niveis["único"] + 0.002 else "único"
print(f"Acurácia balanceada média dos níveis: {bal_niveis} → regra escolhida: {REGRA}")

#### 5. Teste (uma única vez)

Compara o modelo atual (limiar único) com a configuração escolhida. A
diferença de AUC tem intervalo de confiança por bootstrap pareado: as
mesmas reamostragens do teste para os dois modelos.

In [ ]:
cal_atual = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(prob_va["atual"], y_va)
p_atual = cal_atual.predict(modelos["atual"].predict_proba(teste[CAT + NUM])[:, 1])
p_nova = calibrador.predict(modelos[ESCOLHIDA].predict_proba(teste[CAT + VARIANTES[ESCOLHIDA]])[:, 1])

rng = np.random.default_rng(SEMENTE)
difs = []
for _ in range(1000):
    i = rng.integers(0, len(y_te), len(y_te))
    difs.append(roc_auc_score(y_te[i], p_nova[i]) - roc_auc_score(y_te[i], p_atual[i]))
dif_ic = np.percentile(difs, [2.5, 97.5])
print(f"AUC teste · atual: {roc_auc_score(y_te, p_atual):.4f} · {ESCOLHIDA}: {roc_auc_score(y_te, p_nova):.4f}")
print(f"Diferença: {roc_auc_score(y_te, p_nova) - roc_auc_score(y_te, p_atual):+.4f} "
      f"(IC 95%: {dif_ic[0]:+.4f} a {dif_ic[1]:+.4f})")
print(f"Brier · atual: {brier_score_loss(y_te, p_atual):.4f} · {ESCOLHIDA}: {brier_score_loss(y_te, p_nova):.4f}")

final = tabela_decisoes(teste, y_te, p_nova)
print(f"\nDecisões no teste ({ESCOLHIDA}):")
print(final.round(3).to_string())
final.to_csv(PASTA_RES / "experimentos_decisoes_teste.csv")
comparacao.to_csv(PASTA_RES / "experimentos_atributos_validacao.csv")

#### 6. Salvar a configuração final

Só substitui o modelo da demonstração se algo mudou.

In [ ]:
if ESCOLHIDA != "atual" or REGRA != "único":
    joblib.dump({"modelo": modelos[ESCOLHIDA], "calibrador": calibrador,
                 "limiar": LIMIAR_UNICO if REGRA == "único" else LIMIARES_NIVEL,
                 "regra_limiar": REGRA, "colunas": CAT + VARIANTES[ESCOLHIDA],
                 "variante": ESCOLHIDA, "hiperparametros": PARAMS,
                 "medianas_idade": medianas if "idade_relativa" in VARIANTES[ESCOLHIDA] else None},
                PASTA_RES / "modelo_final.joblib")
    print("Salvo resultados/modelo_final.joblib")
else:
    print("Nada mudou: a demonstração continua com resultados/modelo_principal.joblib")

### 6.2 Limiar por nível e por renda

### 18. Dá para reduzir os falsos alarmes de quem não declara a renda?

No teste, o modelo final alerta 40,3% dos estudantes que NÃO declararam renda
e concluíram o curso (falsos alarmes), contra 21,3% de quem declarou. Três
alternativas são comparadas com o modelo final:

- **Sem indicadores:** o mesmo modelo, retreinado sem os dois indicadores de
  não declaração (renda e cor/raça), com calibração e limiares por nível
  escolhidos na validação como no final.
- **Limiar por nível e renda:** o modelo final com quatro limiares (nível ×
  declarou renda?), cada um maximizando a acurácia balanceada na validação.
- **Falsos alarmes iguais:** o modelo final; para quem não declarou renda, o
  limiar de cada nível é o menor que deixa a taxa de falsos alarmes na
  validação no máximo igual à de quem declarou.

**Critério, definido antes de rodar (na validação):** adotar a alternativa se
(1) a diferença de falsos alarmes entre os grupos de renda cair pelo menos
5 pontos, (2) a acurácia balanceada geral não cair mais de 1 ponto e
(3) a detecção de evasões de quem não declarou não ficar abaixo da de quem
declarou. Se mais de uma passar, fica a de maior acurácia balanceada.
O teste só é relatado.

In [ ]:
import glob
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

PASTA_DADOS, PASTA_RES = Path("dados/processados"), Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}
INDICADORES = ["renda_nao_declarada", "cor_nao_declarada"]
GRADE = np.linspace(0.05, 0.95, 91)
NIVEIS = ["Técnico", "Graduação"]


def caminho(pasta, nome):
    """Arquivo gerado pelo notebook; numa sessão nova do Kaggle, procura nas entradas."""
    if (pasta / nome).exists():
        return pasta / nome
    achados = sorted(glob.glob(f"/kaggle/input/**/{nome}", recursive=True))
    if not achados:
        raise SystemExit(f"Não achei {nome}. Rode o notebook antes ou adicione a saída dele como entrada.")
    print(nome, "->", achados[0])
    return Path(achados[0])


final = joblib.load(caminho(PASTA_RES, "modelo_final.joblib"))
pre = final["modelo"].named_steps["pre"]
CAT, NUM = list(pre.transformers_[0][2]), list(pre.transformers_[1][2])
LIMIAR = final["limiar"]

df = pd.read_csv(caminho(PASTA_DADOS, "ef_2021_2023.csv"),
                 dtype={c: str for c in set(CAT) | {"cor_raca", "renda_familiar", "tipo_de_curso"}})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
df["nivel"] = np.where(df["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
partes = {p: df[df["particao"] == p].reset_index(drop=True) for p in ["treino", "validacao", "teste"]}

#### Probabilidades das variantes

In [ ]:
prob = {"final": {p: final["calibrador"].predict(final["modelo"].predict_proba(partes[p][CAT + NUM])[:, 1])
                  for p in ["validacao", "teste"]}}

NUM_SEM = [c for c in NUM if c not in INDICADORES]
sem = clone(final["modelo"])
sem.set_params(pre__transformers=[pre.transformers[0], ("num", "passthrough", NUM_SEM)])
print("Treinando o modelo sem indicadores…")
sem.fit(partes["treino"][CAT + NUM_SEM], partes["treino"]["evadiu"])
bruto_va = sem.predict_proba(partes["validacao"][CAT + NUM_SEM])[:, 1]
calib = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(bruto_va, partes["validacao"]["evadiu"])
prob["sem indicadores"] = {p: calib.predict(sem.predict_proba(partes[p][CAT + NUM_SEM])[:, 1])
                           for p in ["validacao", "teste"]}

#### Limiares (sempre escolhidos na validação)

In [ ]:
va = partes["validacao"]
y_va = va["evadiu"].to_numpy()
nd_va = va["renda_nao_declarada"].to_numpy() == 1


def melhor_bal(y, p):
    return float(GRADE[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in GRADE]))])


def fpr(y, alerta):
    return alerta[y == 0].mean()


limiares = {}
# Final: limiares do próprio modelo final, por nível.
limiares["final"] = {(n, r): LIMIAR[n] for n in NIVEIS for r in [0, 1]}
# Sem indicadores: limiar por nível, como no final.
p = prob["sem indicadores"]["validacao"]
lim_sem = {n: melhor_bal(y_va[va["nivel"] == n], p[va["nivel"] == n]) for n in NIVEIS}
limiares["sem indicadores"] = {(n, r): lim_sem[n] for n in NIVEIS for r in [0, 1]}
# Limiar por nível e renda.
p = prob["final"]["validacao"]
limiares["limiar por nível e renda"] = {
    (n, r): melhor_bal(y_va[(va["nivel"] == n) & (nd_va == r)], p[(va["nivel"] == n) & (nd_va == r)])
    for n in NIVEIS for r in [0, 1]}
# Falsos alarmes iguais.
lim_ig = {}
for n in NIVEIS:
    m_dec, m_nd = (va["nivel"] == n).to_numpy() & ~nd_va, (va["nivel"] == n).to_numpy() & nd_va
    alvo = fpr(y_va[m_dec], p[m_dec] >= LIMIAR[n])
    lim_ig[(n, 0)] = LIMIAR[n]
    lim_ig[(n, 1)] = next((float(t) for t in GRADE
                           if t >= LIMIAR[n] and fpr(y_va[m_nd], p[m_nd] >= t) <= alvo), 0.95)
limiares["falsos alarmes iguais"] = lim_ig
prob["limiar por nível e renda"] = prob["final"]
prob["falsos alarmes iguais"] = prob["final"]

print("Limiares (nível, 0 = declarou renda, 1 = não declarou):")
for nome, l in limiares.items():
    print(f"  {nome}: " + ", ".join(f"{n}/{r}: {t:.2f}" for (n, r), t in l.items()))

#### Comparação

In [ ]:
def resumo(parte, nome):
    d, pp = partes[parte], prob[nome][parte]
    y, nd = d["evadiu"].to_numpy(), d["renda_nao_declarada"].to_numpy()
    lim = np.array([limiares[nome][(n, r)] for n, r in zip(d["nivel"], nd)])
    a = (pp >= lim).astype(int)
    linha = {"AUC": roc_auc_score(y, pp), "acurácia balanceada": balanced_accuracy_score(y, a) * 100}
    for r, rot in [(0, "declarou"), (1, "não declarou")]:
        m = nd == r
        linha[f"evasão real ({rot})"] = y[m].mean() * 100
        linha[f"alertas ({rot})"] = a[m].mean() * 100
        linha[f"detecção ({rot})"] = a[m & (y == 1)].mean() * 100
        linha[f"falsos alarmes ({rot})"] = a[m & (y == 0)].mean() * 100
    linha["diferença de falsos alarmes"] = linha["falsos alarmes (não declarou)"] - linha["falsos alarmes (declarou)"]
    return linha


VARIANTES = ["final", "sem indicadores", "limiar por nível e renda", "falsos alarmes iguais"]
tab = {parte: pd.DataFrame({n: resumo(parte, n) for n in VARIANTES}).T for parte in ["validacao", "teste"]}
pd.set_option("display.width", 250)
print("Validação (base da decisão):")
print(tab["validacao"].round(3).to_string())

b = tab["validacao"].loc["final"]
aprovadas = []
for n in VARIANTES[1:]:
    v = tab["validacao"].loc[n]
    ok = (v["diferença de falsos alarmes"] <= b["diferença de falsos alarmes"] - 5
          and v["acurácia balanceada"] >= b["acurácia balanceada"] - 1
          and v["detecção (não declarou)"] >= v["detecção (declarou)"])
    print(f"{n}: {'passa' if ok else 'não passa'} no critério")
    if ok:
        aprovadas.append(n)
escolha = max(aprovadas, key=lambda n: tab["validacao"].loc[n, "acurácia balanceada"]) if aprovadas else "final"
print(f"\nDecisão: {escolha}")

print("\nTeste (só para relatar):")
print(tab["teste"].round(3).to_string())

PASTA_RES.mkdir(exist_ok=True)
pd.concat(tab).to_csv(PASTA_RES / "alarmes_renda.csv")

#### Regra adotada

Se a alternativa escolhida for "limiar por nível e renda", os quatro limiares
passam a fazer parte do modelo final e são usados daqui em diante (seções
seguintes do notebook e site). O modelo e as probabilidades não mudam.

In [ ]:
if escolha == "limiar por nível e renda":
    final["limiar_renda"] = {f"{n}|{r}": round(t, 2) for (n, r), t in limiares[escolha].items()}
    final["regra_limiar"] = "por nível e renda"
    joblib.dump(final, PASTA_RES / "modelo_final.joblib")
    print("Limiares por nível e renda salvos em resultados/modelo_final.joblib:", final["limiar_renda"])
else:
    print("Modelo final mantido com limiares por nível.")

### 6.3 Nível de confiança

### Onde o modelo é confiável: acerto por nível de confiança

O acerto geral (cerca de 71%) mistura alunos sobre os quais o modelo tem
certeza com alunos em que ele está em dúvida. Aqui o acerto é medido em
função da confiança, no **teste**, com o modelo e os limiares finais
(por nível de curso e por declaração de renda, seção 6.2). Nada é ajustado.

Confiança de uma previsão = distância entre a probabilidade e o limiar do
nível do aluno. Quanto mais longe do limiar, mais "decidido" o modelo está.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PASTA_FIG, PASTA_RES = Path("figuras"), Path("resultados")
COR, CINZA = "#2b6cb0", "#9aa5b1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}


def limiar_por_aluno(d, final):
    """Limiar de alerta de cada aluno: por nível e, se adotado (seção 6.2), por declaração de renda."""
    if final.get("limiar_renda"):
        nd = d["renda_nao_declarada"].astype(int)
        return np.array([final["limiar_renda"][f"{n}|{r}"] for n, r in zip(d["nivel"], nd)])
    lim = final["limiar"]
    return d["nivel"].map(lim).to_numpy() if isinstance(lim, dict) else np.full(len(d), float(lim))


final = joblib.load(PASTA_RES / "modelo_final.joblib")
limiares = final["limiar"]
p = pd.read_csv(PASTA_RES / "predicoes_teste.csv")
p["nivel"] = np.where(p["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
p["renda_nao_declarada"] = (p["renda_familiar"].isin(AUSENTES) | p["renda_familiar"].isna()).astype(int)
p["limiar"] = limiar_por_aluno(p, final)
p["alerta"] = (p["prob_principal"] >= p["limiar"]).astype(int)
p["acertou"] = (p["alerta"] == p["evadiu"]).astype(int)
p["confianca"] = (p["prob_principal"] - p["limiar"]).abs()
print(f"Limiares: {final.get('limiar_renda') or limiares}")
print(f"Acerto geral no teste: {p['acertou'].mean():.1%} em {len(p):,} matrículas")

#### 1. Acerto por faixa de probabilidade

Para cada faixa: quantos alunos caem nela, quanto evadiram de fato e
quanto o modelo acertou.

In [ ]:
faixas = pd.cut(p["prob_principal"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True,
                labels=["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"])
por_faixa = (p.groupby(faixas, observed=True)
               .agg(alunos=("evadiu", "size"), evasao_real=("evadiu", "mean"),
                    acerto=("acertou", "mean"))
               .assign(proporcao=lambda t: t["alunos"] / len(p)))
print((por_faixa * [1, 100, 100, 100]).round(1).rename(columns={
    "evasao_real": "evasão real (%)", "acerto": "acerto (%)", "proporcao": "% dos alunos"}).to_string())

#### 2. Acerto contra cobertura

Ordena os alunos do mais confiante para o menos confiante. Para cada
cobertura (fração dos alunos sobre os quais o modelo "decide"), mostra o
acerto nesse grupo. Os demais ficariam para avaliação humana.

In [ ]:
ordem = p.sort_values("confianca", ascending=False).reset_index(drop=True)
coberturas = np.arange(0.1, 1.01, 0.1)
curva = pd.DataFrame({
    "cobertura": coberturas,
    "alunos": [int(round(c * len(ordem))) for c in coberturas],
    "acerto": [ordem["acertou"].iloc[:int(round(c * len(ordem)))].mean() for c in coberturas],
    "confiança mínima": [ordem["confianca"].iloc[int(round(c * len(ordem))) - 1] for c in coberturas],
})
print((curva.assign(cobertura=curva["cobertura"] * 100, acerto=curva["acerto"] * 100)
            .round(3).to_string(index=False)))
curva.to_csv(PASTA_RES / "acerto_por_cobertura.csv", index=False)

for alvo in [0.80, 0.85, 0.90]:
    ok = curva[curva["acerto"] >= alvo]
    if len(ok):
        print(f"Acerto ≥ {alvo:.0%}: o modelo pode decidir sobre {ok['cobertura'].max():.0%} dos alunos")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curva["cobertura"] * 100, curva["acerto"] * 100, "o-", color=COR)
for _, l in curva.iterrows():
    ax.annotate(f"{l['acerto']:.0%}", (l["cobertura"] * 100, l["acerto"] * 100),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.axhline(p["acertou"].mean() * 100, ls="--", lw=1, color=CINZA)
ax.text(12, p["acertou"].mean() * 100 - 2.5, f"acerto com todos os alunos: {p['acertou'].mean():.0%}",
        fontsize=8, color="#555")
ax.set_xlim(5, 105)
ax.set_ylim(60, 100)
ax.set_xlabel("% dos alunos sobre os quais o modelo decide (os mais confiantes primeiro)")
ax.set_ylabel("Acerto nesse grupo (%)")
ax.set_title("Quanto mais confiante, mais o modelo acerta")
fig.tight_layout()
fig.savefig(PASTA_FIG / "41_acerto_por_cobertura.png", bbox_inches="tight")
plt.show()

#### 3. Quem fica na zona de dúvida

Os 30% menos confiantes: de que nível e faixa etária são.

In [ ]:
duvida = ordem.iloc[int(0.7 * len(ordem)):]
comparacao = pd.DataFrame({
    "todos (%)": p["nivel"].value_counts(normalize=True) * 100,
    "zona de dúvida (%)": duvida["nivel"].value_counts(normalize=True) * 100}).round(1)
print(comparacao.to_string())
print(f"\nIdade mediana no ingresso · todos: {p['idade_ingresso'].median():.0f} · "
      f"zona de dúvida: {duvida['idade_ingresso'].median():.0f}")

### 6.4 Atendimento por capacidade e alunos de até 17 anos

### 19. Onde o modelo vai pior, e o que dá para fazer

Duas perguntas práticas para a assistência estudantil:

1. **Se a equipe só consegue atender os K% de maior risco**, quantos deles
   realmente evadiriam (precisão), e que parte de todas as evasões eles cobrem?
2. **Estudantes de até 17 anos** (sobretudo técnico integrado): poucas
   evasões são detectadas. Um limiar próprio para eles, escolhido na validação
   pela acurácia balanceada, melhora? A que custo em falsos alarmes?

Critério para o limiar por idade, definido antes de rodar (na validação):
adotar se a detecção do grupo subir pelo menos 10 pontos e a acurácia
balanceada geral não cair. O teste só é relatado.

In [ ]:
import glob
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

PASTA_DADOS, PASTA_RES = Path("dados/processados"), Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}
GRADE = np.linspace(0.05, 0.95, 91)



def limiar_por_aluno(d, final):
    """Limiar de alerta de cada aluno: por nível e, se adotado (seção 6.2), por declaração de renda."""
    if final.get("limiar_renda"):
        nd = d["renda_nao_declarada"].astype(int)
        return np.array([final["limiar_renda"][f"{n}|{r}"] for n, r in zip(d["nivel"], nd)])
    lim = final["limiar"]
    return d["nivel"].map(lim).to_numpy() if isinstance(lim, dict) else np.full(len(d), float(lim))


def caminho(pasta, nome):
    if (pasta / nome).exists():
        return pasta / nome
    achados = sorted(glob.glob(f"/kaggle/input/**/{nome}", recursive=True))
    if not achados:
        raise SystemExit(f"Não achei {nome}. Rode o notebook antes ou adicione a saída dele como entrada.")
    return Path(achados[0])


final = joblib.load(caminho(PASTA_RES, "modelo_final.joblib"))
pre = final["modelo"].named_steps["pre"]
CAT, NUM = list(pre.transformers_[0][2]), list(pre.transformers_[1][2])
LIMIAR = final["limiar"]

df = pd.read_csv(caminho(PASTA_DADOS, "ef_2021_2023.csv"),
                 dtype={c: str for c in set(CAT) | {"cor_raca", "renda_familiar", "tipo_de_curso"}})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
df["nivel"] = np.where(df["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
partes = {p: df[df["particao"] == p].reset_index(drop=True) for p in ["validacao", "teste"]}
for d in partes.values():
    d["prob"] = final["calibrador"].predict(final["modelo"].predict_proba(d[CAT + NUM])[:, 1])
    d["jovem"] = d["idade_ingresso"] <= 17

#### 1. Atender só os K% de maior risco

In [ ]:
te = partes["teste"]
linhas = []
for nivel, d in [("Todos", te)] + list(te.groupby("nivel")):
    ordem = d.sort_values("prob", ascending=False)
    for k in [5, 10, 20, 30]:
        top = ordem.head(int(len(ordem) * k / 100))
        linhas.append({"nível": nivel, "K (%)": k, "alunos atendidos": len(top),
                       "precisão (%)": top["evadiu"].mean() * 100,
                       "evasões cobertas (%)": top["evadiu"].sum() / d["evadiu"].sum() * 100,
                       "evasão média do nível (%)": d["evadiu"].mean() * 100})
topk = pd.DataFrame(linhas)
print("Teste — atender os K% de maior risco:")
print(topk.round(1).to_string(index=False))

#### 2. Limiar próprio para estudantes de até 17 anos

In [ ]:
def melhor_bal(y, p):
    return float(GRADE[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int)) for t in GRADE]))])


va = partes["validacao"]
LIM_JOVEM = {n: melhor_bal(va.loc[va["jovem"] & (va["nivel"] == n), "evadiu"],
                           va.loc[va["jovem"] & (va["nivel"] == n), "prob"])
             for n in ["Técnico", "Graduação"] if (va["jovem"] & (va["nivel"] == n)).sum() >= 100}
print("Limiares próprios para até 17 anos:", LIM_JOVEM, "| atuais:", LIMIAR)


def alerta(d, com_jovem):
    lim = limiar_por_aluno(d, final).astype(float)
    if com_jovem:
        for n, t in LIM_JOVEM.items():
            lim[(d["jovem"] & (d["nivel"] == n)).to_numpy()] = t
    return (d["prob"].to_numpy() >= lim).astype(int)


def resumo(d, com_jovem):
    a, y, j = alerta(d, com_jovem), d["evadiu"].to_numpy(), d["jovem"].to_numpy()
    return {"acurácia balanceada (%)": balanced_accuracy_score(y, a) * 100,
            "AUC até 17": roc_auc_score(y[j], d["prob"].to_numpy()[j]),
            "evasão real até 17 (%)": y[j].mean() * 100,
            "detecção até 17 (%)": a[j & (y == 1)].mean() * 100,
            "falsos alarmes até 17 (%)": a[j & (y == 0)].mean() * 100,
            "detecção 18+ (%)": a[~j & (y == 1)].mean() * 100,
            "falsos alarmes 18+ (%)": a[~j & (y == 0)].mean() * 100}


tab = pd.concat({p: pd.DataFrame({"atual": resumo(d, False), "limiar próprio até 17": resumo(d, True)}).T
                 for p, d in partes.items()})
print(tab.round(3).to_string())
v = tab.loc["validacao"]
ok = (v.loc["limiar próprio até 17", "detecção até 17 (%)"] >= v.loc["atual", "detecção até 17 (%)"] + 10
      and v.loc["limiar próprio até 17", "acurácia balanceada (%)"] >= v.loc["atual", "acurácia balanceada (%)"])
print("Decisão:", "ADOTAR limiar próprio para até 17 anos" if ok else "manter limiares atuais")

PASTA_RES.mkdir(exist_ok=True)
topk.to_csv(PASTA_RES / "atendimento_top_k.csv", index=False)
tab.to_csv(PASTA_RES / "limiar_ate_17.csv")

## 7. Renda e cor/raça

### 7.1 Registro no ingresso e indicadores de não declaração

### Renda e cor/raça: de onde vem o sinal e se dá para usar

O modelo com renda e cor/raça ganhou 0,018 de AUC, mas há suspeita de que
parte disso venha do "não declarado" refletir o desfecho (o aluno sai antes
de a instituição completar o cadastro). Este script testa isso de três jeitos:

1. **Quando a renda é preenchida.** No arquivo de Matrículas 2023, que tem
   alunos ainda em curso, compara a falta de renda entre quem acabou de
   entrar e quem entrou há anos. Se quem acabou de entrar já tem renda, ela
   é coletada no ingresso e o "não declarado" é informação legítima.
2. **Dentro de cada instituição.** Se o "não declarado" evade mais mesmo
   comparando alunos da mesma instituição, o sinal não é só diferença entre
   instituições que preenchem mais ou menos.
3. **Separar valor e ausência.** Treina variantes do modelo para ver quanto
   do ganho vem dos valores declarados e quanto vem só de saber quem não
   declarou. Avaliação na validação; o teste não é usado.

Uso, na raiz do repositório: python scripts/10_renda_raca.py

In [ ]:
import re
import unicodedata
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEMENTE = 42
ARQ_MATRICULAS = Path("dados/brutos/microdados_matriculas_2023.csv")
ARQ_BASE = Path("dados/processados/ef_2021_2023.csv")
PASTA_FIG, PASTA_RES = Path("figuras"), Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
NORDESTE = {"AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"}
TIPOS = {"Técnico", "Tecnologia", "Licenciatura", "Bacharelado"}
COR, CINZA = "#2b6cb0", "#9aa5b1"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})


def normaliza(nome):
    sem_acento = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sem_acento.lower()).strip("_")

#### 1. Quando a renda e a cor/raça são preenchidas (Matrículas 2023)

In [ ]:
if ARQ_MATRICULAS.exists():
    with open(ARQ_MATRICULAS, encoding="latin-1") as f:
        cab = f.readline()
    sep = max([";", "\t", ","], key=cab.count)
    quero = {"categoria_da_situacao", "data_de_inicio_do_ciclo", "renda_familiar", "cor_raca",
             "tipo_de_curso", "uf"}
    for enc in ("utf-8-sig", "latin-1"):
        try:
            m = pd.read_csv(ARQ_MATRICULAS, sep=sep, encoding=enc, dtype=str,
                            usecols=lambda c: normaliza(c) in quero)
            break
        except UnicodeDecodeError:
            continue
    m.columns = [normaliza(c) for c in m.columns]
    m = m[m["uf"].str.upper().isin(NORDESTE) & m["tipo_de_curso"].isin(TIPOS)].copy()
    m["ano_ingresso"] = pd.to_datetime(m["data_de_inicio_do_ciclo"], dayfirst=True, errors="coerce").dt.year
    m["situacao"] = m["categoria_da_situacao"].str.lower().map(
        lambda s: "Em curso" if str(s).startswith("em curso") else
                  "Evadiu" if str(s).startswith("evadid") else "Concluiu")
    m["renda_ausente"] = m["renda_familiar"].isin(AUSENTES) | m["renda_familiar"].isna()
    m["cor_ausente"] = m["cor_raca"].isin(AUSENTES) | m["cor_raca"].isna()
    m = m[m["ano_ingresso"].between(2015, 2023)]
    print(f"Matrículas 2023, Nordeste, técnico e graduação: {len(m):,}")

    tabela = (m.groupby(["ano_ingresso", "situacao"])
                .agg(n=("renda_ausente", "size"), renda_ausente=("renda_ausente", "mean"),
                     cor_ausente=("cor_ausente", "mean"))
                .reset_index())
    tabela = tabela[tabela["n"] >= 200]
    print("\n% sem renda e sem cor/raça declaradas, por ano de ingresso e situação em 2023:")
    print(tabela.assign(renda_ausente=lambda t: (t["renda_ausente"] * 100).round(1),
                        cor_ausente=lambda t: (t["cor_ausente"] * 100).round(1)).to_string(index=False))

    em_curso = tabela[tabela["situacao"] == "Em curso"]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.plot(em_curso["ano_ingresso"], em_curso["renda_ausente"] * 100, "o-", color=COR, label="Renda")
    ax.plot(em_curso["ano_ingresso"], em_curso["cor_ausente"] * 100, "o-", color=CINZA, label="Cor/raça")
    ax.set_xlabel("Ano de ingresso (alunos ainda em curso em 2023)")
    ax.set_ylabel("% sem a informação")
    ax.set_ylim(0, 100)
    ax.set_title("Quem acabou de entrar já tem renda e cor/raça registradas?")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(PASTA_FIG / "50_preenchimento_por_ingresso.png", bbox_inches="tight")
    plt.show()
    print("\nComo ler: se a falta de renda entre os ingressantes de 2023 for parecida com a dos anos\n"
          "anteriores, a renda é registrada no ingresso. Se for muito maior e cair com o tempo de curso,\n"
          "ela é completada depois, e o 'não declarado' carrega informação sobre o desfecho.")
else:
    print(f"{ARQ_MATRICULAS} não encontrado: a parte 1 foi pulada.")

#### 2. O "não declarado" evade mais dentro da mesma instituição?

Só treino, instituições com pelo menos 300 matrículas e pelo menos 30
alunos em cada grupo (declarou / não declarou).

In [ ]:
CAT = ["instituicao", "unidade_de_ensino", "uf", "nome_de_curso", "tipo_de_curso", "tipo_de_oferta",
       "modalidade_de_ensino", "eixo_tecnologico", "subeixo_tecnologico", "turno",
       "fonte_de_financiamento", "sexo"]
EXTRA = ["cor_raca", "renda_familiar"]
NUM = ["carga_horaria", "duracao_prevista_meses", "mes_ingresso", "idade_ingresso"]
df = pd.read_csv(ARQ_BASE, dtype={c: str for c in CAT + EXTRA})
df["renda_ausente"] = df["renda_familiar"].isin(AUSENTES).astype(int)
df["cor_ausente"] = df["cor_raca"].isin(AUSENTES).astype(int)
treino = df[df["particao"] == "treino"].reset_index(drop=True)
validacao = df[df["particao"] == "validacao"].reset_index(drop=True)

linhas = []
for inst, g in treino.groupby("instituicao"):
    for campo in ["renda_ausente", "cor_ausente"]:
        a, d = g[g[campo] == 1], g[g[campo] == 0]
        if len(g) >= 300 and len(a) >= 30 and len(d) >= 30:
            linhas.append({"instituicao": inst, "campo": campo, "n": len(g),
                           "% sem informação": a.shape[0] / len(g) * 100,
                           "evasão sem informação": a["evadiu"].mean() * 100,
                           "evasão com informação": d["evadiu"].mean() * 100})
por_inst = pd.DataFrame(linhas)
por_inst["diferença (pontos)"] = por_inst["evasão sem informação"] - por_inst["evasão com informação"]
for campo, nome in [("renda_ausente", "renda"), ("cor_ausente", "cor/raça")]:
    t = por_inst[por_inst["campo"] == campo]
    if len(t):
        print(f"\n{nome}: {len(t)} instituições comparáveis · em {(t['diferença (pontos)'] > 0).sum()} delas "
              f"quem não declarou evade mais · diferença mediana de {t['diferença (pontos)'].median():.1f} pontos")
        print(f"  % sem {nome} por instituição: de {t['% sem informação'].min():.0f}% a {t['% sem informação'].max():.0f}%")
por_inst.round(1).to_csv(PASTA_RES / "renda_raca_por_instituicao.csv", index=False)

#### 3. De onde vem o ganho: valores declarados ou ausência?

Variantes, todas com os hiperparâmetros do modelo principal:

- **principal**: sem renda e cor/raça;
- **+ indicadores de ausência**: principal mais dois campos sim/não (declarou renda? declarou cor/raça?);
- **completo**: principal mais renda e cor/raça, com "não declarado" como categoria;
- **declarados**: treinado e avaliado só em quem declarou as duas; compara o principal com o
  principal mais renda e cor/raça, o que mede o sinal dos valores sem a ausência no meio.

In [ ]:
PARAMS = joblib.load(PASTA_RES / "modelo_principal.joblib")["hiperparametros"]


def pipeline(cats, nums):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), cats),
        ("num", "passthrough", nums)])
    return Pipeline([("pre", pre), ("modelo", HistGradientBoostingClassifier(
        categorical_features=list(range(len(cats))), max_iter=1000, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=30, l2_regularization=1.0,
        random_state=SEMENTE, **PARAMS))])


def avaliar(nome, cats, nums, tr, va):
    modelo = pipeline(cats, nums).fit(tr[cats + nums], tr["evadiu"])
    p = modelo.predict_proba(va[cats + nums])[:, 1]
    return {"variante": nome, "matrículas de validação": len(va),
            "AUC": roc_auc_score(va["evadiu"], p), "Brier": brier_score_loss(va["evadiu"], p)}


resultados = [
    avaliar("principal", CAT, NUM, treino, validacao),
    avaliar("+ indicadores de ausência", CAT, NUM + ["renda_ausente", "cor_ausente"], treino, validacao),
    avaliar("completo", CAT + EXTRA, NUM, treino, validacao),
]
decl_tr = treino[(treino["renda_ausente"] == 0) & (treino["cor_ausente"] == 0)]
decl_va = validacao[(validacao["renda_ausente"] == 0) & (validacao["cor_ausente"] == 0)]
resultados += [
    avaliar("declarados: principal", CAT, NUM, decl_tr, decl_va),
    avaliar("declarados: + renda e cor/raça", CAT + EXTRA, NUM, decl_tr, decl_va),
]
tabela = pd.DataFrame(resultados).set_index("variante")
print(tabela.round(4).to_string())
tabela.to_csv(PASTA_RES / "renda_raca_variantes.csv")

ganho_total = tabela.loc["completo", "AUC"] - tabela.loc["principal", "AUC"]
ganho_ausencia = tabela.loc["+ indicadores de ausência", "AUC"] - tabela.loc["principal", "AUC"]
ganho_valores = tabela.loc["declarados: + renda e cor/raça", "AUC"] - tabela.loc["declarados: principal", "AUC"]
print(f"\nGanho do completo sobre o principal: {ganho_total:+.4f}")
print(f"Ganho só com os indicadores de ausência: {ganho_ausencia:+.4f} "
      f"({ganho_ausencia / ganho_total:.0%} do ganho do completo)" if ganho_total > 0 else "")
print(f"Ganho dos valores declarados, entre quem declarou: {ganho_valores:+.4f}")

### 7.2 Cor/raça como entrada

### Cor/raça como entrada do modelo: corrige o erro com estudantes indígenas?

O modelo principal subestima em cerca de 10 pontos o risco de estudantes
indígenas, porque não vê a cor/raça. Aqui se treina o mesmo modelo com a
cor/raça como entrada e se compara, grupo por grupo:
- calibração (probabilidade média prevista contra evasão real);
- AUC dentro do grupo;
- AUC geral.

A decisão é tomada na validação. O teste aparece no fim só para relatar,
e o intervalo de confiança da evasão real mostra quanto os grupos pequenos
deixam de margem.

Uso, na raiz do repositório: python scripts/11_cor_raca_entrada.py

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEMENTE = 42
PASTA_RES = Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
CAT = ["instituicao", "unidade_de_ensino", "uf", "nome_de_curso", "tipo_de_curso", "tipo_de_oferta",
       "modalidade_de_ensino", "eixo_tecnologico", "subeixo_tecnologico", "turno",
       "fonte_de_financiamento", "sexo"]
NUM = ["carga_horaria", "duracao_prevista_meses", "mes_ingresso", "idade_ingresso",
       "renda_nao_declarada", "cor_nao_declarada"]

df = pd.read_csv("dados/processados/ef_2021_2023.csv", dtype={c: str for c in CAT + ["cor_raca", "renda_familiar"]})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
partes = {p: df[df["particao"] == p].reset_index(drop=True) for p in ["treino", "validacao", "teste"]}
PARAMS = joblib.load(PASTA_RES / "modelo_principal.joblib")["hiperparametros"]
print("Hiperparâmetros do modelo principal:", PARAMS)


def pipeline(cats):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), cats),
        ("num", "passthrough", NUM)])
    return Pipeline([("pre", pre), ("modelo", HistGradientBoostingClassifier(
        categorical_features=list(range(len(cats))), max_iter=1000, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=30, l2_regularization=1.0,
        random_state=SEMENTE, **PARAMS))])

#### Treino e calibração

In [ ]:
VARIANTES = {"principal": CAT, "principal + cor/raça": CAT + ["cor_raca"]}
prob = {}
for nome, cats in VARIANTES.items():
    modelo = pipeline(cats).fit(partes["treino"][cats + NUM], partes["treino"]["evadiu"])
    bruto_va = modelo.predict_proba(partes["validacao"][cats + NUM])[:, 1]
    calib = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(bruto_va, partes["validacao"]["evadiu"])
    for p in ["validacao", "teste"]:
        prob[(nome, p)] = calib.predict(modelo.predict_proba(partes[p][cats + NUM])[:, 1])


def wilson(k, n, z=1.96):
    q = k / n
    centro = (q + z**2 / (2 * n)) / (1 + z**2 / n)
    meia = z * np.sqrt(q * (1 - q) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - meia, centro + meia


def por_grupo(parte):
    d = partes[parte]
    linhas = []
    for grupo, idx in d.groupby("cor_raca").groups.items():
        g = d.loc[idx]
        inf, sup = wilson(g["evadiu"].sum(), len(g))
        linha = {"grupo": grupo, "n": len(g), "evasão real (%)": g["evadiu"].mean() * 100,
                 "IC 95% real": f"{inf * 100:.0f}–{sup * 100:.0f}"}
        for nome in VARIANTES:
            p = prob[(nome, parte)][idx]
            linha[f"previsto · {nome} (%)"] = p.mean() * 100
            linha[f"erro · {nome} (pontos)"] = (p.mean() - g["evadiu"].mean()) * 100
            if g["evadiu"].nunique() == 2 and g["evadiu"].sum() >= 30 and (1 - g["evadiu"]).sum() >= 30:
                linha[f"AUC · {nome}"] = roc_auc_score(g["evadiu"], p)
        linhas.append(linha)
    return pd.DataFrame(linhas).set_index("grupo").sort_values("n", ascending=False)

#### Validação: base da decisão

In [ ]:
geral = pd.DataFrame({
    nome: {"AUC": roc_auc_score(partes["validacao"]["evadiu"], prob[(nome, "validacao")]),
           "Brier": brier_score_loss(partes["validacao"]["evadiu"], prob[(nome, "validacao")])}
    for nome in VARIANTES}).T
print(geral.round(4).to_string())
va = por_grupo("validacao")
print("\nPor cor/raça (validação):")
print(va.round(1).to_string())

erro_ind = {nome: va.loc["Indígena", f"erro · {nome} (pontos)"] for nome in VARIANTES} if "Indígena" in va.index else {}
pior = {nome: va[f"erro · {nome} (pontos)"].abs().max() for nome in VARIANTES}
ganho_auc = geral.loc["principal + cor/raça", "AUC"] - geral.loc["principal", "AUC"]
print(f"\nErro com indígenas · principal: {erro_ind.get('principal', np.nan):+.1f} pontos · "
      f"com cor/raça: {erro_ind.get('principal + cor/raça', np.nan):+.1f} pontos")
print(f"Maior erro entre grupos · principal: {pior['principal']:.1f} · com cor/raça: {pior['principal + cor/raça']:.1f}")
print(f"Diferença de AUC geral: {ganho_auc:+.4f}")
melhora = (erro_ind and abs(erro_ind["principal + cor/raça"]) < abs(erro_ind["principal"]) - 2
           and ganho_auc > -0.002)
print("\nCritério: adotar se o erro com indígenas cair mais de 2 pontos sem perder mais de 0,002 de AUC.")
print("→ ADOTAR cor/raça como entrada" if melhora else "→ NÃO há ganho suficiente para adotar")

#### Teste: só para relatar

In [ ]:
te = por_grupo("teste")
print("Por cor/raça (teste):")
print(te.round(1).to_string())
print({nome: round(roc_auc_score(partes["teste"]["evadiu"], prob[(nome, "teste")]), 4) for nome in VARIANTES})
pd.concat({"validacao": va, "teste": te}).round(3).to_csv(PASTA_RES / "cor_raca_entrada.csv")

### 7.3 Faixa de renda como entrada

### Faixa de renda como entrada do modelo: melhora a calibração por renda?

O modelo final já usa se a renda foi declarada (`renda_nao_declarada`),
mas não a faixa declarada. No script 10, a faixa não melhorou a AUC.
Aqui ela é avaliada com o mesmo tipo de critério usado para a cor/raça
(script 11): justiça entre grupos, não só desempenho geral.

Critério, fixado ANTES de rodar:
  adotar a faixa de renda como entrada se o maior erro de calibração
  (|previsto médio − evasão real|) entre as faixas de renda com pelo menos
  100 estudantes na validação cair mais de 2 pontos, sem perder mais de
  0,002 de AUC geral.

A decisão é tomada na validação. O teste aparece no fim só para relatar.

Uso, na raiz do repositório: python scripts/12_renda_entrada.py

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEMENTE = 42
N_MINIMO = 100
PASTA_RES = Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
CAT = ["instituicao", "unidade_de_ensino", "uf", "nome_de_curso", "tipo_de_curso", "tipo_de_oferta",
       "modalidade_de_ensino", "eixo_tecnologico", "subeixo_tecnologico", "turno",
       "fonte_de_financiamento", "sexo", "cor_raca"]
NUM = ["carga_horaria", "duracao_prevista_meses", "mes_ingresso", "idade_ingresso",
       "renda_nao_declarada", "cor_nao_declarada"]

df = pd.read_csv("dados/processados/ef_2021_2023.csv", dtype={c: str for c in CAT + ["renda_familiar"]})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
# Um só rótulo para "não declarada", para os grupos ficarem comparáveis.
df["faixa_renda"] = df["renda_familiar"].where(df["renda_nao_declarada"] == 0, "Não declarada")
partes = {p: df[df["particao"] == p].reset_index(drop=True) for p in ["treino", "validacao", "teste"]}

try:
    PARAMS = joblib.load(PASTA_RES / "modelo_final.joblib")["hiperparametros"]
except FileNotFoundError:
    PARAMS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20}
print("Hiperparâmetros do modelo final:", PARAMS)


def pipeline(cats):
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), cats),
        ("num", "passthrough", NUM)])
    return Pipeline([("pre", pre), ("modelo", HistGradientBoostingClassifier(
        categorical_features=list(range(len(cats))), max_iter=1000, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=30, l2_regularization=1.0,
        random_state=SEMENTE, **PARAMS))])

#### Treino e calibração

In [ ]:
VARIANTES = {"final": CAT, "final + faixa de renda": CAT + ["faixa_renda"]}
prob = {}
for nome, cats in VARIANTES.items():
    modelo = pipeline(cats).fit(partes["treino"][cats + NUM], partes["treino"]["evadiu"])
    bruto_va = modelo.predict_proba(partes["validacao"][cats + NUM])[:, 1]
    calib = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(bruto_va, partes["validacao"]["evadiu"])
    for p in ["validacao", "teste"]:
        prob[(nome, p)] = calib.predict(modelo.predict_proba(partes[p][cats + NUM])[:, 1])
    print(f"{nome}: {modelo.named_steps['modelo'].n_iter_} árvores")


def wilson(k, n, z=1.96):
    q = k / n
    centro = (q + z**2 / (2 * n)) / (1 + z**2 / n)
    meia = z * np.sqrt(q * (1 - q) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - meia, centro + meia


def por_grupo(parte):
    d = partes[parte]
    linhas = []
    for grupo, idx in d.groupby("faixa_renda").groups.items():
        g = d.loc[idx]
        inf, sup = wilson(g["evadiu"].sum(), len(g))
        linha = {"grupo": grupo, "n": len(g), "evasão real (%)": g["evadiu"].mean() * 100,
                 "IC 95% real": f"{inf * 100:.0f}–{sup * 100:.0f}"}
        for nome in VARIANTES:
            p = prob[(nome, parte)][idx]
            linha[f"previsto · {nome} (%)"] = p.mean() * 100
            linha[f"erro · {nome} (pontos)"] = (p.mean() - g["evadiu"].mean()) * 100
            if g["evadiu"].sum() >= 30 and (1 - g["evadiu"]).sum() >= 30:
                linha[f"AUC · {nome}"] = roc_auc_score(g["evadiu"], p)
        linhas.append(linha)
    return pd.DataFrame(linhas).set_index("grupo").sort_values("n", ascending=False)

#### Validação: base da decisão

In [ ]:
geral = pd.DataFrame({
    nome: {"AUC": roc_auc_score(partes["validacao"]["evadiu"], prob[(nome, "validacao")]),
           "Brier": brier_score_loss(partes["validacao"]["evadiu"], prob[(nome, "validacao")])}
    for nome in VARIANTES}).T
print(geral.round(4).to_string())
va = por_grupo("validacao")
print("\nPor faixa de renda (validação):")
print(va.round(1).to_string())

grandes = va[va["n"] >= N_MINIMO]
pior = {nome: grandes[f"erro · {nome} (pontos)"].abs().max() for nome in VARIANTES}
grupo_pior = grandes["erro · final (pontos)"].abs().idxmax()
ganho_auc = geral.loc["final + faixa de renda", "AUC"] - geral.loc["final", "AUC"]
print(f"\nMaior erro entre faixas com n ≥ {N_MINIMO} · final: {pior['final']:.1f} pontos ({grupo_pior}) · "
      f"com faixa de renda: {pior['final + faixa de renda']:.1f} pontos")
print(f"Diferença de AUC geral: {ganho_auc:+.4f}")
adotar = pior["final + faixa de renda"] < pior["final"] - 2 and ganho_auc > -0.002
print("\nCritério: adotar se o maior erro cair mais de 2 pontos sem perder mais de 0,002 de AUC.")
print("→ ADOTAR a faixa de renda como entrada" if adotar else "→ NÃO há ganho suficiente para adotar")

#### Teste: só para relatar

In [ ]:
te = por_grupo("teste")
print("Por faixa de renda (teste):")
print(te.round(1).to_string())
print({nome: round(roc_auc_score(partes["teste"]["evadiu"], prob[(nome, "teste")]), 4) for nome in VARIANTES})
pd.concat({"validacao": va, "teste": te}).round(3).to_csv(PASTA_RES / "renda_entrada.csv")

### 7.4 Incerteza da calibração por cor/raça

### 16. Incerteza do erro de calibração por cor/raça

O erro de calibração dos estudantes indígenas mudou de −3,0 pontos na validação
para −6,6 no teste. Aqui se mede quanto disso pode ser acaso: para cada grupo de
cor/raça, o erro (risco médio previsto − evasão observada, em pontos) recebe um
intervalo de confiança de 95% por bootstrap. Nada no modelo muda.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

SEMENTE, REPETICOES = 42, 2000
PASTA_DADOS, PASTA_RES = Path("dados/processados"), Path("resultados")

final = joblib.load(PASTA_RES / "modelo_final.joblib")
pre = final["modelo"].named_steps["pre"]
CAT, NUM = list(pre.transformers_[0][2]), list(pre.transformers_[1][2])
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}

df = pd.read_csv(PASTA_DADOS / "ef_2021_2023.csv")
numericas = {"carga_horaria", "ano_ingresso", "mes_ingresso", "duracao_prevista_meses", "idade_ingresso",
             "ano_base", "evadiu", "retido", "inscritos", "vagas", "inscritos_por_vaga"}
categoricas = [c for c in df.columns if c not in numericas]
df[categoricas] = df[categoricas].astype(str)
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
df["nivel"] = np.where(df["tipo_de_curso"].isin({"Tecnologia", "Licenciatura", "Bacharelado"}), "Graduação", "Técnico")

rng = np.random.default_rng(SEMENTE)
linhas = []
for particao in ("validacao", "teste"):
    d = df[df["particao"] == particao].reset_index(drop=True)
    p = final["calibrador"].predict(final["modelo"].predict_proba(d[CAT + NUM])[:, 1])
    y = d["evadiu"].to_numpy()
    for grupo, idx in d.groupby("cor_raca").groups.items():
        idx = np.asarray(idx)
        if len(idx) < 30:
            continue
        erro = (p[idx].mean() - y[idx].mean()) * 100
        amostras = rng.choice(idx, size=(REPETICOES, len(idx)), replace=True)
        boot = (p[amostras].mean(axis=1) - y[amostras].mean(axis=1)) * 100
        ic_inf, ic_sup = np.percentile(boot, [2.5, 97.5])
        linhas.append({"partição": particao, "cor/raça": grupo, "n": len(idx),
                       "evasão observada (%)": y[idx].mean() * 100, "risco previsto (%)": p[idx].mean() * 100,
                       "erro (pontos)": erro, "IC 95% inferior": ic_inf, "IC 95% superior": ic_sup,
                       "IC inclui zero": "sim" if ic_inf <= 0 <= ic_sup else "não"})

tabela = pd.DataFrame(linhas).set_index(["partição", "cor/raça"])
print(tabela.round(1).to_string())
tabela.to_csv(PASTA_RES / "ic_calibracao_raca.csv")

### 7.5 Correções para estudantes indígenas

### 17. Dá para corrigir a subestimação com estudantes indígenas?

O modelo final subestima o risco de estudantes indígenas (−3,0 pontos na
validação, −6,6 no teste, com intervalos que encostam no zero). Duas correções
são testadas contra o modelo final:

- **Recalibração por grupo:** soma ao risco previsto de estudantes indígenas
  a diferença média entre evasão real e risco previsto, aprendida na validação.
- **Peso no treino:** retreina o mesmo modelo dando peso 3 ou 10 às matrículas
  de estudantes indígenas, e recalibra na validação como o modelo final.

**Critérios, definidos antes de rodar:**

- **Peso no treino:** adotar se, na validação, (1) o erro absoluto com indígenas
  cair mais de 2 pontos, (2) a AUC geral não perder mais de 0,002 e (3) o maior
  erro absoluto entre os demais grupos de cor/raça não subir mais de 1 ponto.
- **Recalibração por grupo:** o ajuste é a própria média do erro na validação,
  então medi-lo nos mesmos alunos daria erro zero por construção (validação
  cruzada também, porque o erro médio do grupo é um número só). Adotar apenas
  se o intervalo de 95% do erro na validação não incluir zero, isto é, se houver
  evidência de um viés sistemático a corrigir.

O teste só é relatado, para todas as variantes.

In [ ]:
import glob
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

SEMENTE, REPETICOES = 42, 2000
PASTA_DADOS, PASTA_RES = Path("dados/processados"), Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
GRUPO = "Indígena"
PESOS = [3, 10]




def limiar_por_aluno(d, final):
    """Limiar de alerta de cada aluno: por nível e, se adotado (seção 6.2), por declaração de renda."""
    if final.get("limiar_renda"):
        nd = d["renda_nao_declarada"].astype(int)
        return np.array([final["limiar_renda"][f"{n}|{r}"] for n, r in zip(d["nivel"], nd)])
    lim = final["limiar"]
    return d["nivel"].map(lim).to_numpy() if isinstance(lim, dict) else np.full(len(d), float(lim))


def caminho(pasta, nome):
    """Arquivo gerado pelo notebook; numa sessão nova do Kaggle, procura nas entradas."""
    if (pasta / nome).exists():
        return pasta / nome
    achados = sorted(glob.glob(f"/kaggle/input/**/{nome}", recursive=True))
    if not achados:
        raise SystemExit(f"Não achei {nome}. Rode o notebook antes ou adicione a saída dele como entrada.")
    print(nome, "->", achados[0])
    return Path(achados[0])


final = joblib.load(caminho(PASTA_RES, "modelo_final.joblib"))
pre = final["modelo"].named_steps["pre"]
CAT, NUM = list(pre.transformers_[0][2]), list(pre.transformers_[1][2])
LIMIAR = final["limiar"]
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}

df = pd.read_csv(caminho(PASTA_DADOS, "ef_2021_2023.csv"),
                 dtype={c: str for c in set(CAT) | {"cor_raca", "renda_familiar", "tipo_de_curso"}})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
df["nivel"] = np.where(df["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
partes = {p: df[df["particao"] == p].reset_index(drop=True) for p in ["treino", "validacao", "teste"]}
for p, d in partes.items():
    print(f"{p}: {len(d):,} matrículas, {int((d['cor_raca'] == GRUPO).sum())} de estudantes indígenas")

#### Treino das variantes

In [ ]:
def treina(peso_grupo):
    """Mesmo pipeline e hiperparâmetros do modelo final; peso extra para o grupo no treino."""
    tr = partes["treino"]
    pesos = np.where(tr["cor_raca"] == GRUPO, peso_grupo, 1.0)
    modelo = clone(final["modelo"]).fit(tr[CAT + NUM], tr["evadiu"], modelo__sample_weight=pesos)
    bruto_va = modelo.predict_proba(partes["validacao"][CAT + NUM])[:, 1]
    calib = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(bruto_va, partes["validacao"]["evadiu"])
    return {p: calib.predict(modelo.predict_proba(partes[p][CAT + NUM])[:, 1]) for p in ["validacao", "teste"]}


prob = {"final": {p: final["calibrador"].predict(final["modelo"].predict_proba(partes[p][CAT + NUM])[:, 1])
                  for p in ["validacao", "teste"]}}
for w in PESOS:
    print(f"Treinando com peso {w} para estudantes indígenas…")
    prob[f"peso {w}"] = treina(w)

# Recalibração por grupo: ajuste aprendido em toda a validação, aplicado ao teste.
va, base_va = partes["validacao"], prob["final"]["validacao"]
y_va, m_va = va["evadiu"].to_numpy(), va["cor_raca"].to_numpy() == GRUPO
AJUSTE = y_va[m_va].mean() - base_va[m_va].mean()
p_va = base_va.copy()
p_va[m_va] = np.clip(p_va[m_va] + AJUSTE, 0, 1)  # só para referência: erro ~0 por construção
te = partes["teste"]
p_te = prob["final"]["teste"].copy()
m_te = te["cor_raca"].to_numpy() == GRUPO
p_te[m_te] = np.clip(p_te[m_te] + AJUSTE, 0, 1)
prob["recalibração por grupo"] = {"validacao": p_va, "teste": p_te}
print(f"Ajuste aprendido na validação: {AJUSTE * 100:+.1f} pontos no risco de estudantes indígenas")

#### Comparação

In [ ]:
rng = np.random.default_rng(SEMENTE)


def resumo(parte, p, com_ic=False):
    d = partes[parte]
    y, cor = d["evadiu"].to_numpy(), d["cor_raca"].to_numpy()
    lim = limiar_por_aluno(d, final)
    ind = cor == GRUPO
    erros = {g: (p[cor == g].mean() - y[cor == g].mean()) * 100
             for g in pd.unique(cor) if (cor == g).sum() >= 100 and g != GRUPO}
    linha = {"AUC geral": roc_auc_score(y, p),
             "erro indígenas (pontos)": (p[ind].mean() - y[ind].mean()) * 100,
             "evasões de indígenas detectadas (%)": (p[ind & (y == 1)] >= lim[ind & (y == 1)]).mean() * 100,
             "maior erro nos demais grupos (pontos)": max(abs(e) for e in erros.values())}
    if com_ic:
        i = np.flatnonzero(ind)
        amostras = rng.choice(i, size=(REPETICOES, len(i)), replace=True)
        boot = (p[amostras].mean(axis=1) - y[amostras].mean(axis=1)) * 100
        linha["IC 95% erro indígenas"] = "{:+.1f} a {:+.1f}".format(*np.percentile(boot, [2.5, 97.5]))
    return linha


tab_va = pd.DataFrame({nome: resumo("validacao", p["validacao"], com_ic=(nome == "final"))
                       for nome, p in prob.items()}).T
print("Validação (base da decisão; a linha da recalibração tem erro ~0 por construção):")
print(tab_va.round(4).to_string())

b = tab_va.loc["final"]
decisao = {}
inf, sup = (float(x) for x in b["IC 95% erro indígenas"].split(" a "))
decisao["recalibração por grupo"] = not (inf <= 0 <= sup)
print(f"recalibração por grupo: IC do erro na validação {b['IC 95% erro indígenas']} → "
      f"{'ADOTAR' if decisao['recalibração por grupo'] else 'não adotar (sem evidência de viés sistemático)'}")
for nome in [n for n in tab_va.index if n.startswith("peso")]:
    v = tab_va.loc[nome]
    decisao[nome] = (abs(v["erro indígenas (pontos)"]) < abs(b["erro indígenas (pontos)"]) - 2
                     and v["AUC geral"] - b["AUC geral"] >= -0.002
                     and v["maior erro nos demais grupos (pontos)"] <= b["maior erro nos demais grupos (pontos)"] + 1)
    print(f"{nome}: {'ADOTAR' if decisao[nome] else 'não adotar'}")

tab_te = pd.DataFrame({nome: resumo("teste", p["teste"], com_ic=True) for nome, p in prob.items()}).T
print("\nTeste (só para relatar):")
print(tab_te.round(4).to_string())

PASTA_RES.mkdir(exist_ok=True)
pd.concat({"validacao": tab_va, "teste": tab_te}).to_csv(PASTA_RES / "melhora_indigenas.csv")

## 8. Modelo do Nordeste ou modelo só do IFPI?

### Modelo do Nordeste ou modelo só do IFPI?

Pergunta: para usar dentro do Instituto Federal do Piauí, é melhor o modelo
treinado com todo o Nordeste (que já recebe instituição e campus como
entrada) ou um modelo treinado só com matrículas do IFPI?

Modelos comparados, todos com os hiperparâmetros do modelo final:
  A. Nordeste: o modelo final, como está no site.
  B. Só IFPI: treinado no treino do IFPI e calibrado na validação do IFPI.
  C. Nordeste recalibrado no IFPI: as árvores de A, com a calibração
     refeita na validação do IFPI (corrige um eventual desvio de nível).
  D. Nordeste sem instituição e campus: mostra quanto o modelo ganha por
     saber a instituição.

Todos são avaliados só nas matrículas do IFPI do teste. Os limiares de
alerta (um por nível) são escolhidos na validação do IFPI, para todos os
modelos, maximizando a acurácia balanceada. O teste do IFPI é pequeno
(cerca de 2 mil matrículas), então as diferenças vêm com intervalo de
confiança por bootstrap pareado.

Uso, na raiz do repositório, depois do script 06:
    python scripts/14_modelo_ifpi.py

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

SEMENTE = 42
IFPI = "Instituto Federal do Piauí"
PASTA_RES = Path("resultados")
AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}

final = joblib.load(PASTA_RES / "modelo_final.joblib")
pre = final["modelo"].named_steps["pre"]
CAT, NUM = list(pre.transformers_[0][2]), list(pre.transformers_[1][2])
PARAMS = final["hiperparametros"]
print("Atributos:", CAT + NUM)
print("Hiperparâmetros:", PARAMS)

df = pd.read_csv("dados/processados/ef_2021_2023.csv", dtype={c: str for c in CAT + ["renda_familiar"]})
for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
    df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
df["nivel"] = np.where(df["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
partes = {p: df[df["particao"] == p].reset_index(drop=True) for p in ["treino", "validacao", "teste"]}
ifpi = {p: d[d["instituicao"] == IFPI].reset_index(drop=True) for p, d in partes.items()}
print({p: len(d) for p, d in ifpi.items()}, "matrículas do IFPI por partição")
if len(ifpi["teste"]) < 300:
    raise SystemExit(f"Poucas matrículas de '{IFPI}' no teste. Rode antes o script 02 com a padronização de nomes.")


def pipeline(cats):
    etapa = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan,
                               encoded_missing_value=np.nan), cats),
        ("num", "passthrough", NUM)])
    return Pipeline([("pre", etapa), ("modelo", HistGradientBoostingClassifier(
        categorical_features=list(range(len(cats))), max_iter=1000, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=30, l2_regularization=1.0,
        random_state=SEMENTE, **PARAMS))])


def calibrar(bruto, y):
    return IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(bruto, y)

#### Treino dos modelos

In [ ]:
bruto = lambda m, cats, d: m.predict_proba(d[cats + NUM])[:, 1]
prob = {}  # (modelo, partição do IFPI) -> probabilidade calibrada

# A. Nordeste, como no site
mA = final["modelo"]
for p in ["validacao", "teste"]:
    prob[("A. Nordeste", p)] = final["calibrador"].predict(bruto(mA, CAT, ifpi[p]))

# B. Só IFPI
mB = pipeline(CAT).fit(ifpi["treino"][CAT + NUM], ifpi["treino"]["evadiu"])
calB = calibrar(bruto(mB, CAT, ifpi["validacao"]), ifpi["validacao"]["evadiu"])
for p in ["validacao", "teste"]:
    prob[("B. Só IFPI", p)] = calB.predict(bruto(mB, CAT, ifpi[p]))
print(f"B. Só IFPI: {mB.named_steps['modelo'].n_iter_} árvores, treinado com {len(ifpi['treino']):,} matrículas")

# C. Nordeste recalibrado no IFPI
calC = calibrar(bruto(mA, CAT, ifpi["validacao"]), ifpi["validacao"]["evadiu"])
for p in ["validacao", "teste"]:
    prob[("C. Nordeste recalibrado no IFPI", p)] = calC.predict(bruto(mA, CAT, ifpi[p]))

# D. Nordeste sem instituição e campus
CAT_D = [c for c in CAT if c not in ("instituicao", "unidade_de_ensino")]
mD = pipeline(CAT_D).fit(partes["treino"][CAT_D + NUM], partes["treino"]["evadiu"])
calD = calibrar(bruto(mD, CAT_D, partes["validacao"]), partes["validacao"]["evadiu"])
for p in ["validacao", "teste"]:
    prob[("D. Nordeste sem instituição e campus", p)] = calD.predict(bruto(mD, CAT_D, ifpi[p]))

MODELOS = sorted({m for m, _ in prob})

#### Avaliação no teste do IFPI

In [ ]:
rng = np.random.default_rng(SEMENTE)
te, va = ifpi["teste"], ifpi["validacao"]
y = te["evadiu"].to_numpy()
amostras = [rng.integers(0, len(te), len(te)) for _ in range(1000)]


def limiares(nome):
    """Um limiar por nível, escolhido na validação do IFPI."""
    grade = np.linspace(0.05, 0.95, 91)
    saida = {}
    for nivel in ["Técnico", "Graduação"]:
        m = (va["nivel"] == nivel).to_numpy()
        p, yv = prob[(nome, "validacao")][m], va["evadiu"].to_numpy()[m]
        saida[nivel] = float(grade[int(np.argmax([balanced_accuracy_score(yv, (p >= t).astype(int)) for t in grade]))])
    return saida


linhas = []
for nome in MODELOS:
    p = prob[(nome, "teste")]
    lim = limiares(nome)
    alerta = (p >= te["nivel"].map(lim).to_numpy()).astype(int)
    aucs = [roc_auc_score(y[i], p[i]) for i in amostras]
    linha = {"modelo": nome, "AUC": roc_auc_score(y, p),
             "AUC IC": f"{np.percentile(aucs, 2.5):.3f}–{np.percentile(aucs, 97.5):.3f}",
             "Brier": brier_score_loss(y, p),
             "erro de calibração (pontos)": (p.mean() - y.mean()) * 100,
             "acurácia": (alerta == y).mean(), "acurácia balanceada": balanced_accuracy_score(y, alerta),
             "limiar técnico": lim["Técnico"], "limiar graduação": lim["Graduação"]}
    for nivel in ["Técnico", "Graduação"]:
        m = (te["nivel"] == nivel).to_numpy()
        linha[f"erro calib. {nivel.lower()} (pontos)"] = (p[m].mean() - y[m].mean()) * 100
    linhas.append(linha)
tabela = pd.DataFrame(linhas).set_index("modelo")
print(f"Teste do IFPI: {len(te):,} matrículas, evasão real {y.mean():.1%}\n")
print(tabela.round(3).to_string())

print("\nDiferença de AUC em relação ao A (bootstrap pareado, IC 95%):")
pA = prob[("A. Nordeste", "teste")]
for nome in MODELOS:
    if nome == "A. Nordeste":
        continue
    p = prob[(nome, "teste")]
    dif = [roc_auc_score(y[i], p[i]) - roc_auc_score(y[i], pA[i]) for i in amostras]
    print(f"  {nome}: {roc_auc_score(y, p) - roc_auc_score(y, pA):+.4f} "
          f"({np.percentile(dif, 2.5):+.4f} a {np.percentile(dif, 97.5):+.4f})")

tabela.round(4).to_csv(PASTA_RES / "modelo_ifpi.csv")
print("\nSalvo resultados/modelo_ifpi.csv")

## 9. Retidos: análise de sensibilidade

### Retidos: quem ainda estava em curso depois do prazo

A base principal deixa de fora os alunos "Em curso" depois do fim previsto do
ciclo (retidos), porque o desfecho deles ainda não existe. Esta análise de
sensibilidade responde a três perguntas, sem mudar o modelo nem o site:
  1. Quantos são os retidos e onde estão?
  2. Que risco o modelo atual dá a eles? Se for parecido com o de quem
     evadiu, eles se parecem mais com quem sai do que com quem conclui.
  3. Um modelo para "não concluiu no prazo" (evadiu ou ficou retido) acerta
     tanto quanto o atual? E o que se perde na pergunta original?

Antes, gere as bases com retidos (uma vez por edição):
  for ano in 2021 2022 2023; do
    python scripts/01_preprocessa_pnp.py dados/brutos/microdados_eficiencia_academica_$ano.csv \
        --ano $ano --regiao Nordeste --manter-retidos --saida dados/processados/ef_${ano}_com_retidos.csv
  done
Depois, na raiz do repositório:
  python scripts/15_retidos.py

In [ ]:
if TEM_BRUTOS:
    from pathlib import Path

    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.isotonic import IsotonicRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OrdinalEncoder


    SEMENTE = 42
    PASTA_DADOS, PASTA_RES, PASTA_FIG = Path("dados/processados"), Path("resultados"), Path("figuras")
    PASTA_FIG.mkdir(exist_ok=True)
    AUSENTES = {"Não informado", "Não declarada", "Não declarado", "Não Declarada", "nan"}
    GRADUACAO = {"Tecnologia", "Licenciatura", "Bacharelado"}

    final = joblib.load(PASTA_RES / "modelo_final.joblib")
    pre = final["modelo"].named_steps["pre"]
    CAT, NUM = list(pre.transformers_[0][2]), list(pre.transformers_[1][2])
    PARAMS, LIMIAR = final["hiperparametros"], final["limiar"]



    def limiar_por_aluno(d, final):
        """Limiar de alerta de cada aluno: por nível e, se adotado (seção 6.2), por declaração de renda."""
        if final.get("limiar_renda"):
            nd = d["renda_nao_declarada"].astype(int)
            return np.array([final["limiar_renda"][f"{n}|{r}"] for n, r in zip(d["nivel"], nd)])
        lim = final["limiar"]
        return d["nivel"].map(lim).to_numpy() if isinstance(lim, dict) else np.full(len(d), float(lim))


    def prepara(df):
        df = df.copy()
        numericas = {"carga_horaria", "ano_ingresso", "mes_ingresso", "duracao_prevista_meses", "idade_ingresso",
                     "ano_base", "evadiu", "retido", "inscritos", "vagas", "inscritos_por_vaga"}
        categoricas = [c for c in df.columns if c not in numericas]
        df[categoricas] = df[categoricas].astype(str)
        for novo, origem in [("renda_nao_declarada", "renda_familiar"), ("cor_nao_declarada", "cor_raca")]:
            df[novo] = (df[origem].isin(AUSENTES) | df[origem].isna()).astype(int)
        df["nivel"] = np.where(df["tipo_de_curso"].isin(GRADUACAO), "Graduação", "Técnico")
        return df


    principal = prepara(pd.read_csv(PASTA_DADOS / "ef_2021_2023.csv"))
    arquivos = [PASTA_DADOS / f"ef_{ano}_com_retidos.csv" for ano in (2021, 2022, 2023)]
    faltam = [str(a) for a in arquivos if not a.exists()]
    if faltam:
        raise SystemExit(f"Faltam as bases com retidos: {faltam}. Veja o comando no início deste script.")
    com_retidos = prepara(padroniza_nomes(pd.concat([pd.read_csv(a) for a in arquivos], ignore_index=True), verbose=False))
    retidos = com_retidos[com_retidos["retido"] == 1].reset_index(drop=True)
    print(f"Base principal: {len(principal):,} matrículas com desfecho · "
          f"nas bases com retidos, {int((com_retidos['retido'] == 0).sum()):,} com desfecho (devem ser iguais)")

#### 1. Quantos são e onde estão

In [ ]:
if TEM_BRUTOS:
    total = len(principal) + len(retidos)
    print(f"Retidos: {len(retidos):,} ({len(retidos) / total:.1%} das {total:,} matrículas com ciclo encerrado)\n")
    juntos = pd.concat([principal.assign(situacao=np.where(principal["evadiu"] == 1, "Evadiu", "Concluiu")),
                        retidos.assign(situacao="Retido")], ignore_index=True)
    for coluna, titulo in [("ano_base", "edição"), ("nivel", "nível"), ("tipo_de_curso", "tipo de curso"),
                           ("tipo_de_oferta", "forma de oferta")]:
        tab = pd.crosstab(juntos[coluna], juntos["situacao"], normalize="index").mul(100).round(1)
        tab["n"] = juntos[coluna].value_counts()
        print(f"Situação por {titulo} (%):\n{tab.sort_values('n', ascending=False).to_string()}\n")
    perfil = juntos.groupby("situacao").agg(idade_mediana=("idade_ingresso", "median"),
                                             sem_renda=("renda_nao_declarada", "mean"),
                                             sem_cor=("cor_nao_declarada", "mean"))
    perfil[["sem_renda", "sem_cor"]] *= 100
    print("Perfil por situação (idade mediana; % sem renda e sem cor/raça declaradas):")
    print(perfil.round(1).to_string())

#### 2. Que risco o modelo atual dá aos retidos

Os retidos nunca foram vistos pelo modelo. Compara-se com quem evadiu e quem
concluiu na partição de teste, que o modelo também não viu.

In [ ]:
if TEM_BRUTOS:
    def risco(modelo, calibrador, cats, d):
        return calibrador.predict(modelo.predict_proba(d[cats + NUM])[:, 1])


    teste = principal[principal["particao"] == "teste"].copy()
    grupos = {"Concluiu (teste)": teste[teste["evadiu"] == 0], "Evadiu (teste)": teste[teste["evadiu"] == 1],
              "Retido": retidos}
    linhas, probs = [], {}
    for nome, d in grupos.items():
        p = risco(final["modelo"], final["calibrador"], CAT, d)
        probs[nome] = p
        lim = limiar_por_aluno(d, final)
        for nivel in ["Todos", "Técnico", "Graduação"]:
            m = np.ones(len(d), bool) if nivel == "Todos" else (d["nivel"] == nivel).to_numpy()
            if m.sum() == 0:
                continue
            linhas.append({"grupo": nome, "nível": nivel, "n": int(m.sum()), "risco médio (%)": p[m].mean() * 100,
                           "risco mediano (%)": np.median(p[m]) * 100,
                           "recebem alerta (%)": (p[m] >= lim[m]).mean() * 100})
    tab_risco = pd.DataFrame(linhas).set_index(["nível", "grupo"]).sort_index()
    print(tab_risco.round(1).to_string())

    fig, ax = plt.subplots(figsize=(7, 3.6))
    bins = np.linspace(0, 1, 21)
    for (nome, p), cor in zip(probs.items(), ["#1f4f8f", "#e08a2e", "#6b7a90"]):
        ax.hist(p, bins=bins, density=True, histtype="step", lw=2, color=cor, label=f"{nome} (n={len(p):,})".replace(",", "."))
    ax.set_xlabel("Risco de evasão previsto pelo modelo atual")
    ax.set_ylabel("Densidade")
    ax.set_title("Risco previsto: retidos comparados a quem evadiu e a quem concluiu")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(PASTA_FIG / "42_retidos_risco.png", dpi=150)
    plt.close(fig)
    print("\nFigura: figuras/42_retidos_risco.png")

#### 3. Modelo para "não concluiu no prazo"

Os retidos são divididos 70/20/10 do mesmo jeito que a base principal
(estratificado por edição, semente 42) e somados a ela. O rótulo novo é
1 para quem evadiu ou ficou retido. O modelo usa os mesmos atributos e
hiperparâmetros do modelo final.

In [ ]:
if TEM_BRUTOS:
    idx_tr, idx_resto = train_test_split(retidos.index, test_size=0.30, stratify=retidos["ano_base"], random_state=SEMENTE)
    idx_va, idx_te = train_test_split(idx_resto, test_size=1 / 3, stratify=retidos.loc[idx_resto, "ano_base"],
                                      random_state=SEMENTE)
    retidos["particao"] = "treino"
    retidos.loc[idx_va, "particao"] = "validacao"
    retidos.loc[idx_te, "particao"] = "teste"
    base2 = pd.concat([principal, retidos], ignore_index=True)
    base2["nao_concluiu"] = ((base2["evadiu"] == 1) | (base2["retido"].fillna(0) == 1)).astype(int)
    partes2 = {p: base2[base2["particao"] == p].copy() for p in ["treino", "validacao", "teste"]}
    print("Taxa de 'não concluiu no prazo':", {p: f"{d['nao_concluiu'].mean():.1%}" for p, d in partes2.items()})

    pre2 = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan, encoded_missing_value=np.nan), CAT),
        ("num", "passthrough", NUM)])
    modelo2 = Pipeline([("pre", pre2), ("modelo", HistGradientBoostingClassifier(
        categorical_features=list(range(len(CAT))), max_iter=1000, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=30, l2_regularization=1.0, random_state=SEMENTE, **PARAMS))])
    modelo2.fit(partes2["treino"][CAT + NUM], partes2["treino"]["nao_concluiu"])
    cal2 = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(
        modelo2.predict_proba(partes2["validacao"][CAT + NUM])[:, 1], partes2["validacao"]["nao_concluiu"])
    print(f"Modelo 'não concluiu no prazo': {modelo2.named_steps['modelo'].n_iter_} árvores")

    rng = np.random.default_rng(SEMENTE)


    def auc_ic(y, p, n=500):
        y, p = np.asarray(y), np.asarray(p)
        v = []
        for _ in range(n):
            i = rng.integers(0, len(y), len(y))
            if y[i].min() != y[i].max():
                v.append(roc_auc_score(y[i], p[i]))
        return roc_auc_score(y, p), np.percentile(v, 2.5), np.percentile(v, 97.5)


    te2 = partes2["teste"]
    comparacao = []
    for nome_modelo, (m, c) in {"atual (evasão)": (final["modelo"], final["calibrador"]),
                                 "não concluiu no prazo": (modelo2, cal2)}.items():
        for pergunta, dados, y in [("não concluiu no prazo (teste com retidos)", te2, te2["nao_concluiu"]),
                                   ("evasão (teste original)", teste, teste["evadiu"])]:
            auc, inf, sup = auc_ic(y, risco(m, c, CAT, dados))
            comparacao.append({"modelo": nome_modelo, "pergunta avaliada": pergunta, "n": len(dados),
                               "AUC": auc, "IC 95%": f"{inf:.3f}–{sup:.3f}"})
    comparacao = pd.DataFrame(comparacao).set_index(["pergunta avaliada", "modelo"]).sort_index()
    print(comparacao.round(4).to_string())

#### 4. Comparação completa: cada modelo na sua própria pergunta

Modelo de evasão (o do site) no teste original e modelo "não concluiu no
prazo" no teste com retidos. Os limiares de alerta são um por nível, como no
site: os do modelo final para o primeiro e, para o segundo, escolhidos na
validação com retidos, maximizando a acurácia balanceada. Depois, os mesmos
indicadores de justiça do script 05, por grupo.

In [ ]:
if TEM_BRUTOS:
    from sklearn.metrics import balanced_accuracy_score, brier_score_loss

    va2 = partes2["validacao"]
    p_va2 = risco(modelo2, cal2, CAT, va2)
    grade = np.linspace(0.05, 0.95, 91)


    def melhor_limiar(m):
        return float(grade[int(np.argmax([balanced_accuracy_score(va2["nao_concluiu"].to_numpy()[m],
                                                                  (p_va2[m] >= t).astype(int)) for t in grade]))])


    # Mesma regra do modelo de evasão: por nível e, se ela foi adotada, também por declaração de renda.
    LIMIAR2 = {"limiar": {n: melhor_limiar((va2["nivel"] == n).to_numpy()) for n in ["Técnico", "Graduação"]}}
    if final.get("limiar_renda"):
        LIMIAR2["limiar_renda"] = {f"{n}|{r}": melhor_limiar(((va2["nivel"] == n) & (va2["renda_nao_declarada"] == r)).to_numpy())
                                   for n in ["Técnico", "Graduação"] for r in [0, 1]}
    print("Limiares · evasão:", final.get("limiar_renda") or LIMIAR, "· não concluiu no prazo:",
          LIMIAR2.get("limiar_renda") or LIMIAR2["limiar"])

    MODELOS = {
        "Evasão (site)": dict(dados=teste, y=teste["evadiu"].to_numpy(), p=risco(final["modelo"], final["calibrador"], CAT, teste),
                              limiar=final),
        "Não concluiu no prazo": dict(dados=te2, y=te2["nao_concluiu"].to_numpy(), p=risco(modelo2, cal2, CAT, te2),
                                      limiar=LIMIAR2),
    }
    for m in MODELOS.values():
        m["alerta"] = (m["p"] >= limiar_por_aluno(m["dados"], m["limiar"])).astype(int)


    def medidas(y, p, alerta):
        pos, neg = y == 1, y == 0
        return {"n": len(y), "taxa real (%)": y.mean() * 100, "previsto médio (%)": p.mean() * 100,
                "erro de calibração (pontos)": (p.mean() - y.mean()) * 100,
                "taxa de alerta (%)": alerta.mean() * 100,
                "recall (%)": alerta[pos].mean() * 100 if pos.sum() >= 30 else np.nan,
                "falsos positivos (%)": alerta[neg].mean() * 100 if neg.sum() >= 30 else np.nan}


    geral = []
    for nome, m in MODELOS.items():
        linha = {"modelo": nome, "AUC": roc_auc_score(m["y"], m["p"]), "Brier": brier_score_loss(m["y"], m["p"]),
                 "acurácia (%)": (m["alerta"] == m["y"]).mean() * 100,
                 "acurácia balanceada (%)": balanced_accuracy_score(m["y"], m["alerta"]) * 100}
        linha.update(medidas(m["y"], m["p"], m["alerta"]))
        geral.append(linha)
    geral = pd.DataFrame(geral).set_index("modelo")
    print("\nDesempenho geral, cada modelo na própria pergunta:")
    print(geral.round(3).T.to_string())

    for d in (teste, te2):
        d["faixa_idade"] = pd.cut(d["idade_ingresso"], [0, 17, 20, 24, 29, 120],
                                  labels=["até 17", "18–20", "21–24", "25–29", "30 ou mais"]).astype(str)
        d["renda"] = np.where(d["renda_nao_declarada"] == 1, "não declarou", "declarou")
    GRUPOS = {"nivel": "Nível", "sexo": "Sexo", "cor_raca": "Cor/raça", "renda": "Renda", "faixa_idade": "Idade no ingresso"}
    linhas = []
    for coluna, titulo in GRUPOS.items():
        for nome, m in MODELOS.items():
            d = m["dados"].reset_index(drop=True)
            for grupo, idx in d.groupby(coluna).groups.items():
                if len(idx) < 50 or grupo == "nan":
                    continue
                linhas.append({"atributo": titulo, "grupo": grupo, "modelo": nome,
                               **medidas(m["y"][idx], m["p"][idx], m["alerta"][idx])})
    por_grupo = pd.DataFrame(linhas).set_index(["atributo", "grupo", "modelo"]).sort_index()
    print("\nPor grupo:")
    print(por_grupo.round(1).to_string())

    resumo = (por_grupo.reset_index().groupby(["atributo", "modelo"])
              .agg(maior_erro_calibracao=("erro de calibração (pontos)", lambda s: s.abs().max()),
                   diferenca_recall=("recall (%)", lambda s: s.max() - s.min()),
                   diferenca_falsos_positivos=("falsos positivos (%)", lambda s: s.max() - s.min())))
    print("\nResumo de justiça (pontos percentuais; quanto menor, mais parecido entre grupos):")
    print(resumo.round(1).to_string())

    tab_risco.round(3).to_csv(PASTA_RES / "retidos_risco.csv")
    comparacao.round(4).to_csv(PASTA_RES / "retidos_modelo.csv")
    geral.round(4).to_csv(PASTA_RES / "retidos_comparacao_geral.csv")
    por_grupo.round(4).to_csv(PASTA_RES / "retidos_comparacao_grupos.csv")
    print("\nSalvos em resultados/: retidos_risco.csv, retidos_modelo.csv, retidos_comparacao_geral.csv, "
          "retidos_comparacao_grupos.csv")

## Conclusão

- O gradient boosting com indicadores de "não declarou renda" e "não declarou cor/raça" e com a cor/raça como
  entrada é o modelo final: AUC de 0,829 no teste, bem calibrado.
- Com um limiar por nível, quem não declarou renda recebia muito mais alarmes falsos (40% contra 21%). Limiares
  por nível e por declaração de renda, escolhidos na validação, reduziram essa diferença para 4 pontos, quase sem
  custo na acurácia balanceada (72,4% para 72,3%); essa passou a ser a regra de alerta (seção 6.2).
- As diferenças entre grupos aparecem sobretudo nas taxas de erro (recall e falsos positivos), porque as taxas de
  evasão diferem; a calibração é boa para quase todos os grupos. Estudantes indígenas parecem ter o risco
  subestimado (−3,0 pontos na validação e −6,6 no teste), mas são só 182 e 98 matrículas, e os intervalos de 95%
  incluem ou encostam no zero (seção 7.4).
- Recalibrar esse grupo ou dar a ele peso 3 ou 10 no treino não passou no critério definido antes, na validação.
  No teste, as três correções reduziram o erro para entre −3,7 e −5,1 pontos, dentro da mesma incerteza; escolher
  uma delas pelo teste seria usá-lo para decidir. A saída é ter mais dados, como os da região Norte (seção 7.5).
- Um modelo só do IFPI não supera o regional, que já funciona calibrado dentro do IFPI.
- Os retidos (20% das matrículas com ciclo encerrado) se parecem mais com quem evade; um modelo de "não concluiu
  no prazo" tem desempenho e diferenças entre grupos parecidos com os do modelo de evasão.
- Entre quem entra com até 17 anos, só um terço das evasões é detectado; um limiar próprio para esse grupo
  aumentaria a detecção, mas reduziria a acurácia balanceada geral e não passou no critério (seção 6.4).

O site que usa o modelo final está em https://mingridxs.github.io/evasao-rede-federal-ne/.